# ProductLens 3.0: Stage 2 Dual-Pass Consensus Silver Dataset Labeling
> **Hardware Target:** 2× Kaggle Tesla T4 GPU (Data Parallel Sharding across GPU 0 & GPU 1)  
> **Model:** `Qwen/Qwen3.5-9B` (4-bit via `bitsandbytes`, `dtype=float16`, `language_model_only=True`, `max_model_len=2048`, `max_tokens=384`)  
> **Methodology:** Dual-pass consensus (Pass 1 @ temp 0.0, Pass 2 @ temp 0.2) + Character-Span IoU >= 0.5 Matching + SubstringValidator  
> **Resumability:** Worker-prefixed chunked Parquet checkpointing (`gpu0_batch_*.parquet`, `gpu1_batch_*.parquet`) in `/kaggle/working/stage2_silver_output/`  
> **Target Run:** 5000 reviews (balanced 50/50: 2500 Electronics & Tech / 2500 Apparel & Fashion)

In [ ]:
# [Step 1/5] Install dependencies via uv (fast 1-2 min install)
import subprocess, sys

print("Installing uv and dependencies...")
subprocess.run(
    f"{sys.executable} -m pip install -q uv && uv pip install --system --quiet vllm vllm-bnb-plugin nvidia-cutlass-dsl bitsandbytes pydantic scikit-learn pandas pyarrow",
    shell=True,
    check=True
)
print("✅ Dependencies installed successfully via uv!")

In [ ]:
# [Step 2/5] Hardware & Dynamic CUDA Linker Setup
import os, subprocess, torch

print("Configuring dynamic libcuda.so symlinks for FlashInfer JIT linker...")

candidate_paths = [
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/local/cuda/compat/libcuda.so.1",
    "/usr/local/nvidia/lib64/libcuda.so.1",
    "/usr/local/nvidia/lib/libcuda.so.1",
    "/usr/lib64/libcuda.so.1",
    "/usr/lib/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
    "/usr/local/cuda/lib64/stubs/libcuda.so",
]
libcuda = next((p for p in candidate_paths if os.path.exists(p)), None)

if not libcuda:
    try:
        subprocess.run("ldconfig", shell=True)
        out = subprocess.check_output(
            "ldconfig -p | awk '/libcuda\\.so/ {print $NF; exit}'",
            shell=True,
            text=True,
        ).strip()
        if out and os.path.exists(out):
            libcuda = out
    except Exception:
        pass

if not libcuda:
    try:
        out = subprocess.check_output(
            "find /usr /lib* /usr/local/nvidia -name 'libcuda.so*' 2>/dev/null | head -n 1",
            shell=True,
            text=True,
        ).strip()
        if out and os.path.exists(out):
            libcuda = out
    except Exception:
        pass

assert libcuda and os.path.exists(libcuda), f"libcuda driver library not found! Inspected: {candidate_paths}"
print(f"Found driver library at: {libcuda}")

for dst in [
    "/usr/local/cuda/lib64/stubs/libcuda.so",
    "/usr/local/cuda/lib64/libcuda.so",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
    "/usr/lib/libcuda.so",
]:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.lexists(dst):
        try:
            os.remove(dst)
        except Exception:
            pass
    try:
        os.symlink(libcuda, dst)
    except Exception as e:
        print(f"Symlink warning for {dst}: {e}")

# 1-second fail-fast linker smoke test
subprocess.run(
    "c++ -shared -x c++ /dev/null -L/usr/local/cuda/lib64 -L/usr/local/cuda/lib64/stubs -lcudart -lcuda -o /tmp/test_lcuda.so",
    shell=True,
    check=True,
)
if os.path.exists("/tmp/test_lcuda.so"):
    os.remove("/tmp/test_lcuda.so")
print(f"✅ Linker verified: -lcuda resolved to {libcuda}")

os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# Neutralize unused MiniMax M3 MSA warmup
try:
    import vllm
    from pathlib import Path
    vllm_root = Path(vllm.__file__).parent
    
    mm_file = vllm_root / "model_executor" / "warmup" / "minimax_m3_msa_warmup.py"
    if mm_file.exists():
        mm_file.write_text("def minimax_m3_msa_warmup(*args, **kwargs):\n    pass\n")
        print(f"✅ Neutralized MiniMax M3 warmup at: {mm_file}")
        
    ops_file = vllm_root / "models" / "minimax_m3" / "nvidia" / "ops" / "index_decode_score.py"
    if ops_file.exists():
        ops_file.write_text("def minimax_m3_index_decode_score_cutedsl(*args, **kwargs):\n    pass\n")
        print(f"✅ Neutralized MiniMax M3 ops at: {ops_file}")
except Exception as e:
    print(f"Warmup patch note: {e}")

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("🔑 HF_TOKEN configured from Kaggle Secrets.")
except Exception:
    pass

print(f"CUDA Available: {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"Device Count:   {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory / (1024**3):.2f} GB VRAM)")

In [ ]:
# [Step 3/5] Setup ProductLens 3.0 Core Modules
import os, sys
from pathlib import Path

pkg_dir = Path("/kaggle/working/productlens")
(pkg_dir / "schemas").mkdir(parents=True, exist_ok=True)
(pkg_dir / "validator").mkdir(parents=True, exist_ok=True)
(pkg_dir / "evaluation").mkdir(parents=True, exist_ok=True)
(pkg_dir / "gold").mkdir(parents=True, exist_ok=True)

(pkg_dir / "__init__.py").write_text('"""ProductLens 3.0 package."""\n')
(pkg_dir / "schemas" / "__init__.py").write_text('"""Schemas."""\n')
(pkg_dir / "validator" / "__init__.py").write_text('"""Validator."""\n')
(pkg_dir / "evaluation" / "__init__.py").write_text('"""Evaluation."""\n')
(pkg_dir / "gold" / "__init__.py").write_text('"""Gold."""\n')

(pkg_dir / "schemas" / "enums.py").write_text("\"\"\"\nEnumerations for ProductLens 3.0: Sentiments, Verticals, and Canonical Categories.\n\"\"\"\n\nfrom enum import Enum\nfrom typing import Set\n\n\nclass Sentiment(str, Enum):\n    \"\"\"Aspect-level sentiment polarity.\"\"\"\n    POSITIVE = \"positive\"\n    NEGATIVE = \"negative\"\n    NEUTRAL = \"neutral\"\n    MIXED = \"mixed\"\n\n    @classmethod\n    def from_str(cls, value: str) -> \"Sentiment\":\n        val = value.strip().lower()\n        for member in cls:\n            if member.value == val:\n                return member\n        raise ValueError(f\"Invalid sentiment '{value}'. Allowed: {[m.value for m in cls]}\")\n\n\nclass VerticalDomain(str, Enum):\n    \"\"\"Target e-commerce domains for ProductLens v1/v2.\"\"\"\n    ELECTRONICS_TECH = \"electronics_tech\"\n    APPAREL_FASHION = \"apparel_fashion\"\n\n\nclass ElectronicsCategory(str, Enum):\n    \"\"\"Canonical feature taxonomy for Electronics & Tech.\"\"\"\n    DISPLAY = \"DISPLAY\"\n    BATTERY = \"BATTERY\"\n    SOUND = \"SOUND\"\n    PERFORMANCE = \"PERFORMANCE\"\n    BUILD_QUALITY = \"BUILD_QUALITY\"\n    CONNECTIVITY = \"CONNECTIVITY\"\n    SOFTWARE = \"SOFTWARE\"\n    DELIVERY_INSTALLATION = \"DELIVERY_INSTALLATION\"\n    VALUE = \"VALUE\"\n    OTHER = \"OTHER\"\n\n    @classmethod\n    def values(cls) -> Set[str]:\n        return {item.value for item in cls}\n\n\nclass ApparelCategory(str, Enum):\n    \"\"\"Canonical feature taxonomy for Apparel & Fashion.\"\"\"\n    FABRIC_MATERIAL = \"FABRIC_MATERIAL\"\n    FIT_SIZING = \"FIT_SIZING\"\n    COLOR_PRINT = \"COLOR_PRINT\"\n    STITCHING_DURABILITY = \"STITCHING_DURABILITY\"\n    COMFORT = \"COMFORT\"\n    DELIVERY = \"DELIVERY\"\n    VALUE = \"VALUE\"\n    OTHER = \"OTHER\"\n\n    @classmethod\n    def values(cls) -> Set[str]:\n        return {item.value for item in cls}\n\n\ndef normalize_category(category: str, vertical: VerticalDomain | str) -> str:\n    \"\"\"\n    Validates and normalizes category against the vertical's canonical taxonomy.\n    \"\"\"\n    cat_upper = category.strip().upper().replace(\" \", \"_\")\n    vert = VerticalDomain(vertical) if isinstance(vertical, str) else vertical\n\n    if vert == VerticalDomain.ELECTRONICS_TECH:\n        allowed = ElectronicsCategory.values()\n    elif vert == VerticalDomain.APPAREL_FASHION:\n        allowed = ApparelCategory.values()\n    else:\n        raise ValueError(f\"Unsupported vertical domain: {vertical}\")\n\n    if cat_upper in allowed:\n        return cat_upper\n    return \"OTHER\"\n")
(pkg_dir / "schemas" / "quads.py").write_text("\"\"\"\nQuad definitions and compact serialization for ProductLens 3.0.\n\"\"\"\n\nfrom __future__ import annotations\nfrom typing import Optional\nfrom pydantic import BaseModel, Field, model_validator\nfrom productlens.schemas.enums import Sentiment, VerticalDomain, normalize_category\n\n\nclass RawCompactQuad(BaseModel):\n    \"\"\"\n    Unvalidated quad extracted from compact format or LLM generation.\n    Format: aspect|category|sentiment|confidence|evidence\n    \"\"\"\n    aspect: Optional[str] = None\n    category: str\n    sentiment: Sentiment\n    confidence: float = Field(default=1.0, ge=0.0, le=1.0)\n    evidence: str\n\n    @classmethod\n    def from_compact_line(cls, line: str) -> \"RawCompactQuad\":\n        parts = [p.strip() for p in line.split(\"|\")]\n        if len(parts) < 5:\n            raise ValueError(f\"Malformed compact quad line: '{line}'. Expected 5 pipe-delimited fields.\")\n\n        raw_aspect = parts[0]\n        aspect = None if raw_aspect.lower() in {\"\", \"none\", \"null\"} else raw_aspect\n        category = parts[1]\n        sentiment = Sentiment.from_str(parts[2])\n        try:\n            confidence = float(parts[3])\n        except ValueError:\n            confidence = 1.0\n        evidence = \"|\".join(parts[4:]).strip()  # Allow evidence containing pipes\n\n        return cls(\n            aspect=aspect,\n            category=category,\n            sentiment=sentiment,\n            confidence=max(0.0, min(1.0, confidence)),\n            evidence=evidence\n        )\n\n    def to_compact_line(self) -> str:\n        asp_str = self.aspect if self.aspect is not None else \"null\"\n        return f\"{asp_str}|{self.category}|{self.sentiment.value}|{self.confidence:.2f}|{self.evidence}\"\n\n\nclass AspectQuad(BaseModel):\n    \"\"\"\n    Fully validated ABSA quad with exact character offsets strictly verified in review text.\n    \"\"\"\n    aspect: Optional[str] = Field(default=None, description=\"Aspect term or None for implicit aspects\")\n    category: str = Field(..., description=\"Canonical category (e.g. BATTERY, FABRIC_MATERIAL)\")\n    sentiment: Sentiment = Field(..., description=\"positive, negative, neutral, mixed\")\n    confidence: float = Field(default=1.0, ge=0.0, le=1.0, description=\"Confidence score [0.0, 1.0]\")\n    evidence: str = Field(..., description=\"Exact supporting evidence clause in source text\")\n\n    # Character offsets in the parent review text\n    aspect_start: Optional[int] = Field(default=None, ge=0)\n    aspect_end: Optional[int] = Field(default=None, ge=0)\n    evidence_start: int = Field(..., ge=0)\n    evidence_end: int = Field(..., ge=0)\n\n    @property\n    def is_implicit(self) -> bool:\n        return self.aspect is None\n\n    @model_validator(mode=\"after\")\n    def validate_offsets(self) -> \"AspectQuad\":\n        if self.evidence_end <= self.evidence_start:\n            raise ValueError(\n                f\"Invalid evidence offsets: start ({self.evidence_start}) >= end ({self.evidence_end})\"\n            )\n        if self.aspect is not None:\n            if self.aspect_start is None or self.aspect_end is None:\n                raise ValueError(\"Explicit aspect requires non-null aspect_start and aspect_end offsets.\")\n            if self.aspect_end <= self.aspect_start:\n                raise ValueError(\n                    f\"Invalid aspect offsets: start ({self.aspect_start}) >= end ({self.aspect_end})\"\n                )\n        else:\n            if self.aspect_start is not None or self.aspect_end is not None:\n                raise ValueError(\"Implicit aspect (aspect=None) must have null aspect offsets.\")\n        return self\n\n    def to_compact_line(self) -> str:\n        asp_str = self.aspect if self.aspect is not None else \"null\"\n        return f\"{asp_str}|{self.category}|{self.sentiment.value}|{self.confidence:.2f}|{self.evidence}\"\n")
(pkg_dir / "validator" / "substring.py").write_text("\"\"\"\nDeterministic Substring Locator and Offset Calculator for ProductLens 3.0.\nSolves the offset mismatch / hallucination bug by strictly computing start_char and end_char in code.\nIncludes multi-tier normalized snapping and clause alignment.\n\"\"\"\n\nfrom __future__ import annotations\nimport re\nfrom typing import List, Optional, Tuple\nfrom productlens.schemas.enums import VerticalDomain, normalize_category\nfrom productlens.schemas.quads import AspectQuad, RawCompactQuad\n\n\ndef find_all_occurrences(text: str, sub: str, case_sensitive: bool = True) -> List[Tuple[int, int]]:\n    \"\"\"\n    Finds all [start, end) character spans of sub in text.\n    \"\"\"\n    if not sub:\n        return []\n\n    spans: List[Tuple[int, int]] = []\n    if case_sensitive:\n        start = 0\n        while True:\n            idx = text.find(sub, start)\n            if idx == -1:\n                break\n            spans.append((idx, idx + len(sub)))\n            start = idx + 1\n    else:\n        pattern = re.escape(sub)\n        for match in re.finditer(pattern, text, re.IGNORECASE):\n            spans.append((match.start(), match.end()))\n\n    return spans\n\n\ndef find_normalized_spans(text: str, query: str, is_evidence: bool = False) -> List[Tuple[int, int]]:\n    \"\"\"\n    Finds character spans of query in text with multi-tier deterministic fallback:\n    1. Exact match (case-sensitive)\n    2. Case-insensitive match\n    3. Punctuation/quote-stripped match: strips edge quotes, brackets, and terminal punctuation\n    4. Collapsed whitespace match: regex matching allowing multiple spaces/newlines\n    5. Clause-level substring alignment (if is_evidence=True and evidence spans multiple clauses)\n    \"\"\"\n    if not query:\n        return []\n\n    # Tier 1: Exact case-sensitive\n    spans = find_all_occurrences(text, query, case_sensitive=True)\n    if spans:\n        return spans\n\n    # Tier 2: Case-insensitive\n    spans = find_all_occurrences(text, query, case_sensitive=False)\n    if spans:\n        return spans\n\n    # Tier 3: Edge punctuation / quote stripping\n    clean_query = query.strip(\" \\t\\n\\r\\\"'.,;:!?-\u2013\u2014()[]{}*`\")\n    if clean_query:\n        spans = find_all_occurrences(text, clean_query, case_sensitive=False)\n        if spans:\n            return spans\n\n    # Tier 4: Collapsed whitespace matching\n    if clean_query:\n        words = [re.escape(w) for w in clean_query.split() if w]\n        if len(words) > 1:\n            pattern = r\"\\s+\".join(words)\n            spans = [(m.start(), m.end()) for m in re.finditer(pattern, text, re.IGNORECASE)]\n            if spans:\n                return spans\n\n    # Tier 5: Clause-level alignment for evidence\n    if is_evidence and clean_query:\n        # Split on clause punctuation (., ;, ,)\n        clauses = re.split(r\"[,;.]\\s*\", clean_query)\n        for clause in clauses:\n            c_clean = clause.strip(\" \\t\\n\\r\\\"'.,;:!?-\u2013\u2014()[]{}*`\")\n            words = [re.escape(w) for w in c_clean.split() if w]\n            if len(words) >= 2:\n                pattern = r\"\\s+\".join(words)\n                spans = [(m.start(), m.end()) for m in re.finditer(pattern, text, re.IGNORECASE)]\n                if spans:\n                    return spans\n\n    return []\n\n\nCATEGORY_KEYWORDS = {\n    \"DISPLAY\": [\"display\", \"screen\", \"panel\", \"oled\", \"brightness\", \"refresh rate\", \"resolution\", \"pixel\"],\n    \"BATTERY\": [\"battery\", \"charger\", \"charging\", \"battery backup\", \"backup\", \"power\"],\n    \"SOUND\": [\"sound\", \"audio\", \"speaker\", \"speakers\", \"volume\", \"bass\", \"mic\", \"microphone\"],\n    \"PERFORMANCE\": [\"performance\", \"processor\", \"speed\", \"lag\", \"cpu\", \"gpu\", \"ssd\", \"ram\", \"thermal\", \"heating\"],\n    \"BUILD_QUALITY\": [\"build\", \"quality\", \"construction\", \"body\", \"hinge\", \"chassis\", \"material\", \"finish\"],\n    \"CONNECTIVITY\": [\"connectivity\", \"bluetooth\", \"wifi\", \"ports\", \"port\", \"usb\", \"hdmi\", \"headphone jack\"],\n    \"SOFTWARE\": [\"software\", \"os\", \"app\", \"firmware\", \"windows\", \"driver\", \"ui\"],\n    \"DELIVERY_INSTALLATION\": [\"delivery\", \"installation\", \"shipping\", \"courier\", \"package\"],\n    \"VALUE\": [\"price\", \"value\", \"money\", \"worth\", \"cost\", \"pricing\"],\n    \"FABRIC_MATERIAL\": [\"fabric\", \"material\", \"cloth\", \"cotton\", \"polyester\", \"wool\", \"silk\"],\n    \"FIT_SIZING\": [\"sizing\", \"size\", \"fit\", \"fitting\", \"cut\"],\n    \"COLOR_PRINT\": [\"color\", \"colour\", \"print\", \"shade\", \"dye\", \"pattern\"],\n    \"STITCHING_DURABILITY\": [\"stitching\", \"stitch\", \"durability\", \"seam\", \"thread\", \"dhaaga\"],\n    \"COMFORT\": [\"comfort\", \"comfortable\", \"feel\"],\n    \"DELIVERY\": [\"delivery\", \"shipping\", \"courier\", \"package\"],\n}\n\n\nclass SubstringValidator:\n    \"\"\"\n    Deterministic validator that resolves exact character spans for aspect and evidence.\n    Applies multi-tier normalized snapping, clause alignment, and enum-fallback before hard rejection.\n    \"\"\"\n\n    @classmethod\n    def validate_quad(\n        cls,\n        text: str,\n        raw: RawCompactQuad,\n        vertical: VerticalDomain | str,\n        case_sensitive: bool = True,\n    ) -> Tuple[Optional[AspectQuad], Optional[str]]:\n        \"\"\"\n        Locates evidence and aspect within text, validates containment,\n        and constructs a fully verified AspectQuad.\n        Returns: (AspectQuad, None) on success or (None, error_str) on failure.\n        \"\"\"\n        normalized_category = normalize_category(raw.category, vertical)\n\n        # 0. Enum-to-Noun Fallback: If raw.aspect is an uppercase category tag not present in text,\n        # extract the matching aspect noun from evidence or treat as implicit\n        aspect_query = raw.aspect\n        if aspect_query and aspect_query.isupper() and not find_normalized_spans(text, aspect_query):\n            cat_key = str(raw.category).upper()\n            found_kw = None\n            ev_text = raw.evidence or \"\"\n            for kw in CATEGORY_KEYWORDS.get(cat_key, []):\n                if re.search(r\"\\b\" + re.escape(kw) + r\"\\b\", ev_text, re.IGNORECASE):\n                    found_kw = kw\n                    break\n            if found_kw:\n                aspect_query = found_kw\n            elif cat_key in [\"VALUE\", \"COMFORT\"]:\n                aspect_query = None\n\n        # 1. Locate Evidence (exact, case-insensitive, edge-stripped, or clause-aligned)\n        if case_sensitive:\n            evidence_spans = find_all_occurrences(text, raw.evidence, case_sensitive=True)\n            if not evidence_spans:\n                evidence_spans = find_normalized_spans(text, raw.evidence, is_evidence=True)\n        else:\n            evidence_spans = find_normalized_spans(text, raw.evidence, is_evidence=True)\n\n        if not evidence_spans:\n            return None, f\"Evidence '{raw.evidence}' not found in review text.\"\n\n        # 2. Implicit Aspect case (aspect is None)\n        if aspect_query is None:\n            best_ev_start, best_ev_end = evidence_spans[0]\n            exact_evidence = text[best_ev_start:best_ev_end]\n            quad = AspectQuad(\n                aspect=None,\n                category=normalized_category,\n                sentiment=raw.sentiment,\n                confidence=raw.confidence,\n                evidence=exact_evidence,\n                aspect_start=None,\n                aspect_end=None,\n                evidence_start=best_ev_start,\n                evidence_end=best_ev_end,\n            )\n            return quad, None\n\n        # 3. Explicit Aspect case: locate aspect within evidence span\n        if case_sensitive:\n            aspect_spans = find_all_occurrences(text, aspect_query, case_sensitive=True)\n            if not aspect_spans:\n                aspect_spans = find_normalized_spans(text, aspect_query, is_evidence=False)\n        else:\n            aspect_spans = find_normalized_spans(text, aspect_query, is_evidence=False)\n\n        if not aspect_spans:\n            return None, f\"Aspect '{aspect_query}' not found in review text.\"\n\n        # Disambiguate: Find pair where aspect is strictly inside evidence span\n        matched_pair: Optional[Tuple[Tuple[int, int], Tuple[int, int]]] = None\n        for ev_s, ev_e in evidence_spans:\n            for asp_s, asp_e in aspect_spans:\n                if ev_s <= asp_s and asp_e <= ev_e:\n                    matched_pair = ((asp_s, asp_e), (ev_s, ev_e))\n                    break\n            if matched_pair:\n                break\n\n        # Fallback if aspect and evidence are distinct clauses in review text\n        if matched_pair is None:\n            # Pick first occurrence of each\n            matched_pair = (aspect_spans[0], evidence_spans[0])\n\n        (asp_start, asp_end), (ev_start, ev_end) = matched_pair\n        exact_aspect = text[asp_start:asp_end]\n        exact_evidence = text[ev_start:ev_end]\n\n        try:\n            quad = AspectQuad(\n                aspect=exact_aspect,\n                category=normalized_category,\n                sentiment=raw.sentiment,\n                confidence=raw.confidence,\n                evidence=exact_evidence,\n                aspect_start=asp_start,\n                aspect_end=asp_end,\n                evidence_start=ev_start,\n                evidence_end=ev_end,\n            )\n            return quad, None\n        except Exception as e:\n            return None, f\"Quad construction failed: {str(e)}\"\n")
(pkg_dir / "evaluation" / "metrics.py").write_text("\"\"\"\nEvaluation metrics for ProductLens 3.0:\nExplicit Aspect F1 (multiset Counter), Implicit Category F1, Sentiment Macro-F1 (explicit + implicit),\nOffset Validity, Compact Parse Rate, ECE, and ARTS Accuracy.\n\"\"\"\n\nfrom __future__ import annotations\nfrom collections import Counter\nfrom typing import Dict, List, Optional, Tuple\nimport numpy as np\nfrom sklearn.metrics import f1_score\nfrom productlens.schemas.enums import Sentiment\nfrom productlens.schemas.quads import AspectQuad\n\n\ndef compute_char_iou(span1: Tuple[int, int], span2: Tuple[int, int]) -> float:\n    \"\"\"\n    Computes Character-level Intersection over Union (IoU) between two [start, end) character spans.\n    \"\"\"\n    start1, end1 = span1\n    start2, end2 = span2\n    inter = max(0, min(end1, end2) - max(start1, start2))\n    union = max(end1, end2) - min(start1, start2)\n    return inter / union if union > 0 else 0.0\n\n\ndef compute_explicit_aspect_spans_f1(\n    gold_quads_list: List[List[AspectQuad]],\n    pred_quads_list: List[List[AspectQuad]],\n    iou_threshold: float = 0.5,\n) -> Dict[str, any]:\n    \"\"\"\n    Computes Precision, Recall, and F1 for explicit aspects:\n      1. Strict Exact Offset Matching: multiset matching by (aspect.lower().strip(), start, end)\n      2. Overlap IoU Matching: greedy bipartite matching where character IoU >= iou_threshold\n    \"\"\"\n    tp = 0\n    fp = 0\n    fn = 0\n    overlap_tp = 0\n    total_gold_explicit = 0\n    total_pred_explicit = 0\n\n    for gold_quads, pred_quads in zip(gold_quads_list, pred_quads_list):\n        gold_explicit = [q for q in gold_quads if not q.is_implicit and q.aspect is not None]\n        pred_explicit = [q for q in pred_quads if not q.is_implicit and q.aspect is not None]\n\n        total_gold_explicit += len(gold_explicit)\n        total_pred_explicit += len(pred_explicit)\n\n        # 1. Strict exact offset matching\n        gold_counts = Counter(\n            (q.aspect.lower().strip(), q.aspect_start, q.aspect_end)\n            for q in gold_explicit\n        )\n        pred_counts = Counter(\n            (q.aspect.lower().strip(), q.aspect_start, q.aspect_end)\n            for q in pred_explicit\n        )\n\n        matched_counts = gold_counts & pred_counts\n        matched_in_review = sum(matched_counts.values())\n\n        tp += matched_in_review\n        fp += sum((pred_counts - gold_counts).values())\n        fn += sum((gold_counts - pred_counts).values())\n\n        # 2. Overlap IoU >= 0.5 greedy matching\n        candidate_pairs = []\n        for g_idx, gq in enumerate(gold_explicit):\n            if gq.aspect_start is None or gq.aspect_end is None:\n                continue\n            for p_idx, pq in enumerate(pred_explicit):\n                if pq.aspect_start is None or pq.aspect_end is None:\n                    continue\n                iou = compute_char_iou((gq.aspect_start, gq.aspect_end), (pq.aspect_start, pq.aspect_end))\n                if iou >= iou_threshold:\n                    candidate_pairs.append((iou, g_idx, p_idx))\n\n        candidate_pairs.sort(key=lambda x: x[0], reverse=True)\n        used_gold = set()\n        used_pred = set()\n        for iou, g_idx, p_idx in candidate_pairs:\n            if g_idx not in used_gold and p_idx not in used_pred:\n                used_gold.add(g_idx)\n                used_pred.add(p_idx)\n                overlap_tp += 1\n\n    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0\n\n    overlap_prec = overlap_tp / total_pred_explicit if total_pred_explicit > 0 else 0.0\n    overlap_rec = overlap_tp / total_gold_explicit if total_gold_explicit > 0 else 0.0\n    overlap_f1 = (\n        2 * overlap_prec * overlap_rec / (overlap_prec + overlap_rec)\n        if (overlap_prec + overlap_rec) > 0\n        else 0.0\n    )\n\n    return {\n        \"aspect_precision\": round(precision, 4),\n        \"aspect_recall\": round(recall, 4),\n        \"aspect_f1\": round(f1, 4),\n        \"strict_aspect_f1\": round(f1, 4),\n        \"overlap_aspect_precision\": round(overlap_prec, 4),\n        \"overlap_aspect_recall\": round(overlap_rec, 4),\n        \"overlap_aspect_f1\": round(overlap_f1, 4),\n        \"explicit_matched_count\": tp,\n        \"explicit_overlap_matched_count\": overlap_tp,\n        \"total_gold_explicit\": total_gold_explicit,\n        \"total_pred_explicit\": total_pred_explicit,\n    }\n\n\ndef compute_implicit_category_f1(\n    gold_quads_list: List[List[AspectQuad]],\n    pred_quads_list: List[List[AspectQuad]],\n) -> Dict[str, any]:\n    \"\"\"\n    Computes Precision, Recall, and F1 for implicit aspects (aspect: null)\n    matched by canonical category.\n    \"\"\"\n    tp = 0\n    fp = 0\n    fn = 0\n    total_gold_implicit = 0\n    total_pred_implicit = 0\n\n    for gold_quads, pred_quads in zip(gold_quads_list, pred_quads_list):\n        gold_implicit = [q for q in gold_quads if q.is_implicit]\n        pred_implicit = [q for q in pred_quads if q.is_implicit]\n\n        total_gold_implicit += len(gold_implicit)\n        total_pred_implicit += len(pred_implicit)\n\n        gold_counts = Counter(q.category for q in gold_implicit)\n        pred_counts = Counter(q.category for q in pred_implicit)\n\n        matched_counts = gold_counts & pred_counts\n        matched_in_review = sum(matched_counts.values())\n\n        tp += matched_in_review\n        fp += sum((pred_counts - gold_counts).values())\n        fn += sum((gold_counts - pred_counts).values())\n\n    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0\n\n    return {\n        \"implicit_category_precision\": round(precision, 4),\n        \"implicit_category_recall\": round(recall, 4),\n        \"implicit_category_f1\": round(f1, 4),\n        \"implicit_matched_count\": tp,\n        \"total_gold_implicit\": total_gold_implicit,\n        \"total_pred_implicit\": total_pred_implicit,\n    }\n\n\ndef compute_sentiment_macro_f1(\n    gold_quads_list: List[List[AspectQuad]],\n    pred_quads_list: List[List[AspectQuad]],\n) -> Dict[str, any]:\n    \"\"\"\n    Computes Sentiment Macro-F1 across all matched aspect spans (both explicit and implicit)\n    across all 4 classes: positive, negative, neutral, mixed.\n    \"\"\"\n    y_true: List[str] = []\n    y_pred: List[str] = []\n\n    for gold_quads, pred_quads in zip(gold_quads_list, pred_quads_list):\n        # 1. Match explicit aspects: multiset matching by (aspect.lower(), start, end)\n        pred_explicit_pool = [q for q in pred_quads if not q.is_implicit and q.aspect is not None]\n        for gq in [q for q in gold_quads if not q.is_implicit and q.aspect is not None]:\n            g_key = (gq.aspect.lower().strip(), gq.aspect_start, gq.aspect_end)\n            for idx, pq in enumerate(pred_explicit_pool):\n                p_key = (pq.aspect.lower().strip(), pq.aspect_start, pq.aspect_end)\n                if g_key == p_key:\n                    y_true.append(gq.sentiment.value)\n                    y_pred.append(pq.sentiment.value)\n                    pred_explicit_pool.pop(idx)\n                    break\n\n        # 2. Match implicit aspects: match by category\n        pred_implicit_pool = [q for q in pred_quads if q.is_implicit]\n        for gq in [q for q in gold_quads if q.is_implicit]:\n            for idx, pq in enumerate(pred_implicit_pool):\n                if gq.category == pq.category:\n                    y_true.append(gq.sentiment.value)\n                    y_pred.append(pq.sentiment.value)\n                    pred_implicit_pool.pop(idx)\n                    break\n\n    if not y_true:\n        return {\"sentiment_macro_f1\": 0.0, \"matched_aspect_count\": 0}\n\n    present_classes = sorted(list(set(y_true) | set(y_pred)))\n    macro_f1 = f1_score(y_true, y_pred, labels=present_classes, average=\"macro\", zero_division=0)\n\n    # Per-class F1 for all defined sentiments\n    all_classes = [s.value for s in Sentiment]\n    per_class = f1_score(y_true, y_pred, labels=all_classes, average=None, zero_division=0)\n    per_class_f1 = {f\"f1_{cls_name}\": round(float(score), 4) for cls_name, score in zip(all_classes, per_class)}\n\n    res = {\n        \"sentiment_macro_f1\": round(float(macro_f1), 4),\n        \"matched_aspect_count\": len(y_true),\n    }\n    res.update(per_class_f1)\n    return res\n\n\ndef compute_expected_calibration_error(\n    confidences: List[float],\n    correctness: List[int],\n    n_bins: int = 10,\n) -> float:\n    \"\"\"\n    Computes Expected Calibration Error (ECE) with equal-width confidence bins.\n    \"\"\"\n    if not confidences or not correctness or len(confidences) != len(correctness):\n        return 0.0\n\n    conf_arr = np.array(confidences)\n    corr_arr = np.array(correctness)\n    bin_boundaries = np.linspace(0, 1, n_bins + 1)\n    ece = 0.0\n\n    for i in range(n_bins):\n        bin_lower = bin_boundaries[i]\n        bin_upper = bin_boundaries[i + 1]\n        in_bin = (conf_arr > bin_lower) & (conf_arr <= bin_upper) if i > 0 else (conf_arr >= bin_lower) & (conf_arr <= bin_upper)\n        prop_in_bin = np.mean(in_bin)\n\n        if prop_in_bin > 0:\n            avg_confidence = np.mean(conf_arr[in_bin])\n            avg_accuracy = np.mean(corr_arr[in_bin])\n            ece += np.abs(avg_accuracy - avg_confidence) * prop_in_bin\n\n    return round(float(ece), 4)\n\n\ndef compute_parse_and_offset_rates(\n    total_raw_lines: int,\n    parsed_lines: int,\n    validated_quads: int,\n) -> Dict[str, float]:\n    \"\"\"\n    Calculates parse rate and deterministic offset validity rate.\n    \"\"\"\n    parse_rate = (parsed_lines / total_raw_lines) if total_raw_lines > 0 else 1.0\n    offset_validity_rate = (validated_quads / parsed_lines) if parsed_lines > 0 else 1.0\n\n    return {\n        \"parse_rate\": round(parse_rate, 4),\n        \"offset_validity_rate\": round(offset_validity_rate, 4),\n    }\n")
(pkg_dir / "gold" / "schema.py").write_text("\"\"\"\nGold Set Schema, Partitioning, and Hard Case definitions.\nSupports English and Hinglish (Latin-script Hindi + English) reviews with negation, sarcasm, and implicit aspects.\n\"\"\"\n\nfrom __future__ import annotations\nfrom typing import List, Optional\nfrom pydantic import BaseModel, Field\nfrom productlens.schemas.enums import Sentiment, VerticalDomain\nfrom productlens.schemas.quads import AspectQuad\n\n\nclass GoldReview(BaseModel):\n    \"\"\"A curated, gold-standard review for benchmarking.\"\"\"\n    review_id: str\n    product_id: str\n    text: str\n    vertical: VerticalDomain\n    language: str = Field(default=\"en\", description=\"'en' for English or 'hi-en' for Hinglish\")\n    is_hard_case: bool = False\n    hard_case_type: Optional[str] = Field(\n        default=None,\n        description=\"negation, sarcasm, implicit_aspect, cross_clause_contrast, etc.\"\n    )\n    quads: List[AspectQuad] = Field(default_factory=list)\n\n\nclass GoldSetManifest(BaseModel):\n    \"\"\"Manifest summary for the gold set.\"\"\"\n    total_reviews: int\n    english_reviews: int\n    hinglish_reviews: int\n    hard_cases: int\n    total_quads: int\n    products_count: int\n")
(pkg_dir / "gold" / "manager.py").write_text("\"\"\"\nGold Set Manager: Loads, validates, and partitions the curated gold benchmark set.\n\"\"\"\n\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Dict, List\nfrom productlens.gold.schema import GoldReview, GoldSetManifest\nfrom productlens.schemas.enums import VerticalDomain\n\n\nDEFAULT_GOLD_PATH = Path(__file__).parent / \"gold_dataset.jsonl\"\n\n\nclass GoldSetManager:\n    \"\"\"Manages the curated gold benchmark dataset.\"\"\"\n\n    @staticmethod\n    def load_reviews(path: Path | str = DEFAULT_GOLD_PATH) -> List[GoldReview]:\n        filepath = Path(path)\n        if not filepath.exists():\n            raise FileNotFoundError(f\"Gold set file does not exist at {filepath}\")\n\n        reviews: List[GoldReview] = []\n        with open(filepath, \"r\", encoding=\"utf-8\") as f:\n            for line in f:\n                line = line.strip()\n                if line:\n                    reviews.append(GoldReview.model_validate_json(line))\n        return reviews\n\n    @classmethod\n    def get_manifest(cls, reviews: List[GoldReview]) -> GoldSetManifest:\n        products = {r.product_id for r in reviews}\n        english_count = sum(1 for r in reviews if r.language == \"en\")\n        hinglish_count = sum(1 for r in reviews if r.language == \"hi-en\")\n        hard_count = sum(1 for r in reviews if r.is_hard_case)\n        quad_count = sum(len(r.quads) for r in reviews)\n\n        return GoldSetManifest(\n            total_reviews=len(reviews),\n            english_reviews=english_count,\n            hinglish_reviews=hinglish_count,\n            hard_cases=hard_count,\n            total_quads=quad_count,\n            products_count=len(products),\n        )\n\n    @classmethod\n    def split_by_language(cls, reviews: List[GoldReview]) -> Dict[str, List[GoldReview]]:\n        return {\n            \"en\": [r for r in reviews if r.language == \"en\"],\n            \"hi-en\": [r for r in reviews if r.language == \"hi-en\"],\n        }\n")
(pkg_dir / "evaluation" / "runner.py").write_text("\"\"\"\nStage 1 Evaluation Runner for ProductLens 3.0.\nExecutes benchmarks against gold set, disaggregating English vs. Hinglish and hard cases.\nSupports explicit multiset matching and implicit category tracking.\n\"\"\"\n\nfrom __future__ import annotations\nimport time\nfrom typing import Callable, Dict, List, Optional\nfrom productlens.gold.schema import GoldReview\nfrom productlens.gold.manager import GoldSetManager\nfrom productlens.schemas.quads import AspectQuad\nfrom productlens.evaluation.metrics import (\n    compute_explicit_aspect_spans_f1,\n    compute_implicit_category_f1,\n    compute_sentiment_macro_f1,\n    compute_expected_calibration_error,\n    compute_parse_and_offset_rates,\n)\n\n\nclass Stage1Evaluator:\n    \"\"\"Evaluates ABSA systems and compact parses on the Stage 1 Gold Set.\"\"\"\n\n    def __init__(self, reviews: Optional[List[GoldReview]] = None):\n        self.reviews = reviews if reviews is not None else GoldSetManager.load_reviews()\n\n    def evaluate_predictions(\n        self,\n        predict_fn: Callable[[GoldReview], List[AspectQuad]],\n    ) -> Dict[str, any]:\n        \"\"\"\n        Runs prediction function over the gold set and calculates comprehensive Stage 1 metrics.\n        \"\"\"\n        start_time = time.perf_counter()\n\n        gold_quads_all: List[List[AspectQuad]] = []\n        pred_quads_all: List[List[AspectQuad]] = []\n\n        # Subsets for disaggregated evaluation\n        lang_groups: Dict[str, Dict[str, List[List[AspectQuad]]]] = {\n            \"en\": {\"gold\": [], \"pred\": []},\n            \"hi-en\": {\"gold\": [], \"pred\": []},\n        }\n        hard_case_groups: Dict[str, Dict[str, List[List[AspectQuad]]]] = {\n            \"hard\": {\"gold\": [], \"pred\": []},\n            \"standard\": {\"gold\": [], \"pred\": []},\n        }\n\n        confidences: List[float] = []\n        correctness: List[int] = []\n\n        for r in self.reviews:\n            gold_quads = r.quads\n            pred_quads = predict_fn(r)\n\n            gold_quads_all.append(gold_quads)\n            pred_quads_all.append(pred_quads)\n\n            # Route by language\n            lang_key = r.language if r.language in lang_groups else \"en\"\n            lang_groups[lang_key][\"gold\"].append(gold_quads)\n            lang_groups[lang_key][\"pred\"].append(pred_quads)\n\n            # Route by hard case\n            hard_key = \"hard\" if r.is_hard_case else \"standard\"\n            hard_case_groups[hard_key][\"gold\"].append(gold_quads)\n            hard_case_groups[hard_key][\"pred\"].append(pred_quads)\n\n            # Calibration data: collect on matched explicit spans and implicit categories\n            pred_explicit_pool = [q for q in pred_quads if not q.is_implicit and q.aspect is not None]\n            for gq in [q for q in gold_quads if not q.is_implicit and q.aspect is not None]:\n                g_key = (gq.aspect.lower().strip(), gq.aspect_start, gq.aspect_end)\n                for idx, pq in enumerate(pred_explicit_pool):\n                    p_key = (pq.aspect.lower().strip(), pq.aspect_start, pq.aspect_end)\n                    if g_key == p_key:\n                        confidences.append(pq.confidence)\n                        correctness.append(1 if pq.sentiment == gq.sentiment else 0)\n                        pred_explicit_pool.pop(idx)\n                        break\n\n            pred_implicit_pool = [q for q in pred_quads if q.is_implicit]\n            for gq in [q for q in gold_quads if q.is_implicit]:\n                for idx, pq in enumerate(pred_implicit_pool):\n                    if gq.category == pq.category:\n                        confidences.append(pq.confidence)\n                        correctness.append(1 if pq.sentiment == gq.sentiment else 0)\n                        pred_implicit_pool.pop(idx)\n                        break\n\n        elapsed = time.perf_counter() - start_time\n        throughput_sec_per_500 = (elapsed / len(self.reviews)) * 500 if self.reviews else 0.0\n\n        # Helper to compute composite bundle for any group\n        def evaluate_group(gold_list: List[List[AspectQuad]], pred_list: List[List[AspectQuad]]) -> Dict[str, any]:\n            exp_metrics = compute_explicit_aspect_spans_f1(gold_list, pred_list)\n            imp_metrics = compute_implicit_category_f1(gold_list, pred_list)\n            sent_metrics = compute_sentiment_macro_f1(gold_list, pred_list)\n            return {\n                **exp_metrics,\n                **imp_metrics,\n                **sent_metrics,\n                \"total_matched_quads\": exp_metrics[\"explicit_matched_count\"] + imp_metrics[\"implicit_matched_count\"],\n            }\n\n        overall_metrics = evaluate_group(gold_quads_all, pred_quads_all)\n        ece = compute_expected_calibration_error(confidences, correctness)\n\n        en_metrics = evaluate_group(lang_groups[\"en\"][\"gold\"], lang_groups[\"en\"][\"pred\"])\n        hi_metrics = evaluate_group(lang_groups[\"hi-en\"][\"gold\"], lang_groups[\"hi-en\"][\"pred\"])\n        hard_metrics = evaluate_group(hard_case_groups[\"hard\"][\"gold\"], hard_case_groups[\"hard\"][\"pred\"])\n\n        return {\n            \"total_reviews\": len(self.reviews),\n            \"total_matched_quads\": overall_metrics[\"total_matched_quads\"],\n            \"explicit_matched_count\": overall_metrics[\"explicit_matched_count\"],\n            \"implicit_matched_count\": overall_metrics[\"implicit_matched_count\"],\n            \"overall\": {\n                **overall_metrics,\n                \"expected_calibration_error\": ece,\n            },\n            \"english\": en_metrics,\n            \"hinglish\": hi_metrics,\n            \"hard_cases\": hard_metrics,\n            \"latency\": {\n                \"elapsed_seconds\": round(elapsed, 4),\n                \"seconds_per_500_reviews\": round(throughput_sec_per_500, 4),\n            },\n        }\n")

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

print("✅ ProductLens 3.0 core modules staged successfully!")

In [ ]:
# [Step 4/5] Stage Worker Process Script (stage2_silver_worker.py)
from pathlib import Path

worker_file = Path("/kaggle/working/stage2_silver_worker.py")
worker_file.parent.mkdir(parents=True, exist_ok=True)
worker_file.write_text("#!/usr/bin/env python3\nimport os\nimport sys\nimport json\nimport time\nimport argparse\nfrom pathlib import Path\nfrom typing import Dict, List, Literal, Optional, Set, Tuple\n\nif \"/kaggle/working\" not in sys.path:\n    sys.path.insert(0, \"/kaggle/working\")\n\nos.environ[\"VLLM_USE_FLASHINFER_SAMPLER\"] = \"0\"\n\nimport pandas as pd\nfrom pydantic import BaseModel, Field\n\nfrom vllm import LLM, SamplingParams\ntry:\n    from vllm.sampling_params import StructuredOutputsParams\nexcept ImportError:\n    from vllm.model_executor.guided_decoding.guided_fields import StructuredOutputsParams\n\nfrom productlens.schemas.enums import Sentiment, VerticalDomain\nfrom productlens.schemas.quads import AspectQuad, RawCompactQuad\nfrom productlens.validator.substring import SubstringValidator\n\n# Schema definitions\nCanonicalCategory = Literal[\n    \"DISPLAY\", \"BATTERY\", \"SOUND\", \"PERFORMANCE\", \"BUILD_QUALITY\",\n    \"CONNECTIVITY\", \"SOFTWARE\", \"DELIVERY_INSTALLATION\", \"VALUE\",\n    \"FABRIC_MATERIAL\", \"FIT_SIZING\", \"COLOR_PRINT\", \"STITCHING_DURABILITY\",\n    \"COMFORT\", \"DELIVERY\", \"OTHER\",\n]\n\nclass TeacherAspectQuad(BaseModel):\n    aspect: Optional[str] = Field(None, description=\"Verbatim aspect term or null if implicit\")\n    category: CanonicalCategory = Field(..., description=\"Canonical category\")\n    sentiment: Literal[\"positive\", \"negative\", \"neutral\", \"mixed\"] = Field(..., description=\"Sentiment polarity\")\n    confidence: float = Field(1.0, ge=0.0, le=1.0)\n    evidence: str = Field(..., description=\"Exact substring evidence from review\")\n\nclass TeacherExtractionOutput(BaseModel):\n    quads: List[TeacherAspectQuad] = Field(default_factory=list)\n\nschema_json = json.dumps(TeacherExtractionOutput.model_json_schema())\n\nPASS1_SYSTEM_PROMPT = 'You are an expert Aspect-Based Sentiment Analysis (ABSA) system specialized in e-commerce customer reviews.\\nYour task is to analyze customer feedback and extract fine-grained aspect sentiment quads.\\n\\nStrict Rules:\\n1. \\'aspect\\': MUST BE an exact verbatim substring noun/entity copied directly from the review text (e.g., \"battery life\", \"fabric quality\", \"picture clarity\", \"bass\").\\n   CRITICAL: NEVER set \\'aspect\\' to a category enum name (DO NOT output \"DISPLAY\", \"BATTERY\", \"SOUND\", \"FABRIC_MATERIAL\", \"FIT_SIZING\", \"VALUE\", etc. as the aspect).\\n2. If an aspect is implicit (no specific noun term is named, e.g., \"worth every penny\", \"paisa vasool\", \"terrible purchase\", \"too expensive\"), set \\'aspect\\' to null.\\n3. Map every aspect to the canonical category:\\n   - Electronics: DISPLAY, BATTERY, SOUND, PERFORMANCE, BUILD_QUALITY, CONNECTIVITY, SOFTWARE, DELIVERY_INSTALLATION, VALUE, OTHER.\\n   - Apparel: FABRIC_MATERIAL, FIT_SIZING, COLOR_PRINT, STITCHING_DURABILITY, COMFORT, DELIVERY, VALUE, OTHER.\\n4. Classify sentiment strictly into: \"positive\", \"negative\", \"neutral\", or \"mixed\".\\n5. Evidence MUST be an exact verbatim substring from the review text containing the opinion or rationale. Do not paraphrase or add punctuation.\\n6. Provide calibrated confidence between 0.0 and 1.0.'\nPASS2_SYSTEM_PROMPT = 'You are a precise review intelligence analyst for tech and retail products.\\nDissect the provided user review into feature-level sentiment quads:\\n\\nGuidelines:\\n1. \\'aspect\\': Must be the concrete noun or phrase extracted verbatim from the review text (e.g., \"screen\", \"keyboard\", \"stitching\").\\n   CRITICAL: NEVER use uppercase taxonomy categories (like \"DISPLAY\", \"PERFORMANCE\", \"FABRIC_MATERIAL\", \"VALUE\") in the \\'aspect\\' field. For implied aspects with no concrete noun in text, set \\'aspect\\' to null.\\n2. Assign the canonical category that best matches the product vertical (e.g., DISPLAY, BATTERY, FABRIC_MATERIAL, FIT_SIZING, VALUE).\\n3. Grade sentiment polarity as \"positive\", \"negative\", \"neutral\", or \"mixed\".\\n4. The evidence field MUST be a verbatim quote copied directly from the review text without any modification.\\n5. Assign a confidence score from 0.0 to 1.0.'\nFEW_SHOT_EXAMPLES = [{'role': 'user', 'content': 'Product Domain: electronics_tech\\nReview Text:\\n\"\"\"The OLED screen is incredibly bright and colors pop, but battery life drops quickly under heavy load.\"\"\"\\n\\nExtract all aspect quads as JSON adhering to the schema.'}, {'role': 'assistant', 'content': '{\"quads\": [{\"aspect\": \"OLED screen\", \"category\": \"DISPLAY\", \"sentiment\": \"positive\", \"confidence\": 0.98, \"evidence\": \"The OLED screen is incredibly bright and colors pop\"}, {\"aspect\": \"battery life\", \"category\": \"BATTERY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"battery life drops quickly under heavy load\"}]}'}, {'role': 'user', 'content': 'Product Domain: apparel_fashion\\nReview Text:\\n\"\"\"The cotton fabric feels soft against the skin, but stitching came undone along the collar seam after two washes.\"\"\"\\n\\nExtract all aspect quads as JSON adhering to the schema.'}, {'role': 'assistant', 'content': '{\"quads\": [{\"aspect\": \"cotton fabric\", \"category\": \"FABRIC_MATERIAL\", \"sentiment\": \"positive\", \"confidence\": 0.98, \"evidence\": \"The cotton fabric feels soft against the skin\"}, {\"aspect\": \"stitching\", \"category\": \"STITCHING_DURABILITY\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"stitching came undone along the collar seam after two washes\"}]}'}, {'role': 'user', 'content': 'Product Domain: electronics_tech\\nReview Text:\\n\"\"\"Earphones ka bass dhamakedar hai lekin mic quality calls ke liye bilkul acchi nahi hai.\"\"\"\\n\\nExtract all aspect quads as JSON adhering to the schema.'}, {'role': 'assistant', 'content': '{\"quads\": [{\"aspect\": \"bass\", \"category\": \"SOUND\", \"sentiment\": \"positive\", \"confidence\": 0.98, \"evidence\": \"Earphones ka bass dhamakedar hai\"}, {\"aspect\": \"mic quality\", \"category\": \"SOUND\", \"sentiment\": \"negative\", \"confidence\": 0.95, \"evidence\": \"mic quality calls ke liye bilkul acchi nahi hai\"}]}'}, {'role': 'user', 'content': 'Product Domain: electronics_tech\\nReview Text:\\n\"\"\"Paisa vasool mobile hai bhai, definitely worth every penny.\"\"\"\\n\\nExtract all aspect quads as JSON adhering to the schema.'}, {'role': 'assistant', 'content': '{\"quads\": [{\"aspect\": null, \"category\": \"VALUE\", \"sentiment\": \"positive\", \"confidence\": 0.95, \"evidence\": \"Paisa vasool mobile hai bhai, definitely worth every penny\"}]}'}]\n\ndef compute_char_iou(span1: Tuple[Optional[int], Optional[int]], span2: Tuple[Optional[int], Optional[int]]) -> float:\n    s1, e1 = span1\n    s2, e2 = span2\n    if s1 is None or e1 is None or s2 is None or e2 is None:\n        return 0.0\n    inter_start = max(s1, s2)\n    inter_end = min(e1, e2)\n    intersection = max(0, inter_end - inter_start)\n    len1 = max(0, e1 - s1)\n    len2 = max(0, e2 - s2)\n    union = len1 + len2 - intersection\n    if union <= 0:\n        return 0.0\n    return float(intersection / union)\n\ndef format_prompts(tokenizer, rows, sys_prompt):\n    prompts = []\n    for _, row in rows.iterrows():\n        user_content = (\n            \"Product Domain: \" + str(row[\"vertical\"]) + \"\\n\"\n            \"Review Text:\\n\"\n            '\\\"\\\"\\\"' + str(row[\"text\"]) + '\\\"\\\"\\\"\\n\\n'\n            \"Extract all aspect quads as JSON adhering to the schema.\"\n        )\n        msgs = [\n            {\"role\": \"system\", \"content\": sys_prompt},\n            *FEW_SHOT_EXAMPLES,\n            {\"role\": \"user\", \"content\": user_content},\n        ]\n        p_str = tokenizer.apply_chat_template(\n            msgs,\n            tokenize=False,\n            add_generation_prompt=True,\n            chat_template_kwargs={\"enable_thinking\": False},\n        )\n        prompts.append(p_str)\n    return prompts\n\ndef parse_quads(raw_text):\n    try:\n        data = json.loads(raw_text)\n        return [TeacherAspectQuad.model_validate(q) for q in data.get(\"quads\", [])]\n    except Exception:\n        return []\n\ndef validate_quads(text, t_quads, vertical):\n    validated = []\n    for tq in t_quads:\n        raw_c = RawCompactQuad(\n            aspect=tq.aspect,\n            category=tq.category,\n            sentiment=Sentiment(tq.sentiment),\n            confidence=tq.confidence,\n            evidence=tq.evidence,\n        )\n        v_quad, err = SubstringValidator.validate_quad(text, raw_c, vertical)\n        if v_quad is not None:\n            validated.append(v_quad)\n    return validated\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--worker-id\", type=int, default=0)\n    parser.add_argument(\"--gpu-id\", type=int, default=0)\n    parser.add_argument(\"--shard-file\", type=str, required=True)\n    parser.add_argument(\"--output-dir\", type=str, required=True)\n    parser.add_argument(\"--prefix\", type=str, default=\"gpu0_batch\")\n    parser.add_argument(\"--batch-size\", type=int, default=100)\n    parser.add_argument(\"--max-tokens\", type=int, default=384)\n    args = parser.parse_args()\n\n    print(f\"[Worker {args.worker_id}] Initializing on GPU {args.gpu_id} (PID: {os.getpid()})...\", flush=True)\n    out_dir = Path(args.output_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    df_shard = pd.read_parquet(args.shard_file)\n    print(f\"[Worker {args.worker_id}] Loaded shard {args.shard_file} with {len(df_shard)} reviews.\", flush=True)\n\n    # Resume checkpoint: filter already processed reviews across all worker parquets\n    processed_ids = set()\n    for pf in out_dir.glob(\"*.parquet\"):\n        try:\n            df_p = pd.read_parquet(pf, columns=[\"review_id\"])\n            processed_ids.update(df_p[\"review_id\"].astype(str).tolist())\n        except Exception:\n            pass\n\n    if processed_ids:\n        df_shard = df_shard[~df_shard[\"review_id\"].astype(str).isin(processed_ids)].reset_index(drop=True)\n        print(f\"[Worker {args.worker_id}] Checkpoint filtered: {len(df_shard)} unprocessed reviews remaining.\", flush=True)\n\n    if len(df_shard) == 0:\n        print(f\"[Worker {args.worker_id}] All reviews in shard have already been processed! Exiting.\", flush=True)\n        sys.exit(0)\n\n    # Initialize vLLM\n    MODEL_NAME = \"Qwen/Qwen3.5-9B\"\n    MAX_MODEL_LEN = 2048\n    MAX_NUM_SEQS = 64\n    DTYPE = \"float16\"\n    QUANTIZATION = \"bitsandbytes\"\n\n    print(f\"[Worker {args.worker_id}] Launching vLLM Engine ({MODEL_NAME}, TP=1)...\", flush=True)\n    llm_kwargs = {\n        \"model\": MODEL_NAME,\n        \"tensor_parallel_size\": 1,\n        \"dtype\": DTYPE,\n        \"quantization\": QUANTIZATION,\n        \"load_format\": \"bitsandbytes\",\n        \"trust_remote_code\": True,\n        \"max_model_len\": MAX_MODEL_LEN,\n        \"max_num_seqs\": MAX_NUM_SEQS,\n        \"gpu_memory_utilization\": 0.90,\n        \"language_model_only\": True,\n    }\n    try:\n        llm = LLM(**llm_kwargs)\n    except TypeError as e:\n        err_str = str(e)\n        if \"language_model_only\" in err_str:\n            llm_kwargs.pop(\"language_model_only\", None)\n        if \"load_format\" in err_str:\n            llm_kwargs.pop(\"load_format\", None)\n        llm = LLM(**llm_kwargs)\n\n    tokenizer = llm.get_tokenizer()\n    print(f\"[Worker {args.worker_id}] [OK] vLLM Engine successfully initialized on GPU {args.gpu_id}!\", flush=True)\n\n    sampling_p1 = SamplingParams(\n        temperature=0.0,\n        max_tokens=args.max_tokens,\n        structured_outputs=StructuredOutputsParams(json=schema_json),\n    )\n    sampling_p2 = SamplingParams(\n        temperature=0.2,\n        max_tokens=args.max_tokens,\n        structured_outputs=StructuredOutputsParams(json=schema_json),\n    )\n\n    existing_batches = list(out_dir.glob(f\"{args.prefix}_*.parquet\"))\n    start_batch_idx = max([int(p.stem.split(\"_\")[-1]) for p in existing_batches] + [0]) + 1\n    num_batches = (len(df_shard) + args.batch_size - 1) // args.batch_size\n\n    print(f\"[Worker {args.worker_id}] Processing {len(df_shard)} reviews in {num_batches} batches (start_idx={start_batch_idx})...\", flush=True)\n\n    total_raw_p1 = 0\n    total_raw_p2 = 0\n    total_valid_p1 = 0\n    total_valid_p2 = 0\n    total_consensus_kept = 0\n    total_inference_time = 0.0\n\n    for b in range(num_batches):\n        b_start = b * args.batch_size\n        b_end = min((b + 1) * args.batch_size, len(df_shard))\n        chunk = df_shard.iloc[b_start:b_end]\n\n        t0 = time.perf_counter()\n        prompts_p1 = format_prompts(tokenizer, chunk, PASS1_SYSTEM_PROMPT)\n        outputs_p1 = llm.generate(prompts_p1, sampling_p1)\n\n        prompts_p2 = format_prompts(tokenizer, chunk, PASS2_SYSTEM_PROMPT)\n        outputs_p2 = llm.generate(prompts_p2, sampling_p2)\n        batch_sec = time.perf_counter() - t0\n        total_inference_time += batch_sec\n\n        batch_records = []\n        b_raw_p1 = 0\n        b_raw_p2 = 0\n        b_val_p1 = 0\n        b_val_p2 = 0\n        b_agreed = 0\n\n        for (_, row), o1, o2 in zip(chunk.iterrows(), outputs_p1, outputs_p2):\n            raw1 = parse_quads(o1.outputs[0].text)\n            raw2 = parse_quads(o2.outputs[0].text)\n            b_raw_p1 += len(raw1)\n            b_raw_p2 += len(raw2)\n\n            q1 = validate_quads(row[\"text\"], raw1, row[\"vertical\"])\n            q2 = validate_quads(row[\"text\"], raw2, row[\"vertical\"])\n            b_val_p1 += len(q1)\n            b_val_p2 += len(q2)\n\n            # Relaxed Character-Span IoU >= 0.5 Consensus Matching\n            agreed = []\n            pool2 = list(q2)\n\n            for q1_item in q1:\n                best_idx = None\n                best_iou = -1.0\n\n                for idx, q2_item in enumerate(pool2):\n                    if q1_item.category != q2_item.category or q1_item.sentiment != q2_item.sentiment:\n                        continue\n\n                    if q1_item.aspect is None and q2_item.aspect is None:\n                        best_idx = idx\n                        best_iou = 1.0\n                        break\n                    elif q1_item.aspect is not None and q2_item.aspect is not None:\n                        if (\n                            q1_item.aspect_start is not None\n                            and q1_item.aspect_end is not None\n                            and q2_item.aspect_start is not None\n                            and q2_item.aspect_end is not None\n                        ):\n                            iou = compute_char_iou(\n                                (q1_item.aspect_start, q1_item.aspect_end),\n                                (q2_item.aspect_start, q2_item.aspect_end),\n                            )\n                            if iou >= 0.5 and iou > best_iou:\n                                best_iou = iou\n                                best_idx = idx\n\n                if best_idx is not None:\n                    q2_match = pool2.pop(best_idx)\n                    avg_conf = round(float((q1_item.confidence + q2_match.confidence) / 2.0), 4)\n                    consensus_quad = q1_item.model_copy(update={\"confidence\": avg_conf})\n                    agreed.append(consensus_quad)\n\n            b_agreed += len(agreed)\n\n            for q in agreed:\n                batch_records.append({\n                    \"review_id\": str(row[\"review_id\"]),\n                    \"product_id\": str(row[\"product_id\"]),\n                    \"text\": str(row[\"text\"]),\n                    \"aspect\": q.aspect,\n                    \"category\": q.category,\n                    \"sentiment\": q.sentiment.value if hasattr(q.sentiment, \"value\") else str(q.sentiment),\n                    \"confidence\": q.confidence,\n                    \"evidence\": q.evidence,\n                    \"aspect_start\": q.aspect_start,\n                    \"aspect_end\": q.aspect_end,\n                    \"evidence_start\": q.evidence_start,\n                    \"evidence_end\": q.evidence_end,\n                    \"vertical\": str(row[\"vertical\"]),\n                })\n\n        total_raw_p1 += b_raw_p1\n        total_raw_p2 += b_raw_p2\n        total_valid_p1 += b_val_p1\n        total_valid_p2 += b_val_p2\n        total_consensus_kept += b_agreed\n\n        batch_idx = start_batch_idx + b\n        batch_file = out_dir / f\"{args.prefix}_{batch_idx:04d}.parquet\"\n        if batch_records:\n            df_batch = pd.DataFrame(batch_records)\n        else:\n            df_batch = pd.DataFrame(columns=[\n                \"review_id\", \"product_id\", \"text\", \"aspect\", \"category\",\n                \"sentiment\", \"confidence\", \"evidence\", \"aspect_start\",\n                \"aspect_end\", \"evidence_start\", \"evidence_end\", \"vertical\",\n            ])\n        df_batch.to_parquet(batch_file, index=False)\n\n        p1_pass = (b_val_p1 / b_raw_p1 * 100) if b_raw_p1 > 0 else 0.0\n        p2_pass = (b_val_p2 / b_raw_p2 * 100) if b_raw_p2 > 0 else 0.0\n        cons_surv = (b_agreed / max(1, min(b_val_p1, b_val_p2)) * 100) if min(b_val_p1, b_val_p2) > 0 else 0.0\n\n        print(\n            f\"[Worker {args.worker_id}] [Batch] {batch_file.name} [{b+1}/{num_batches}]: \"\n            f\"{len(chunk)} reviews in {batch_sec:.1f}s ({(len(chunk)/batch_sec):.1f} rev/s) | \"\n            f\"Pass1 Val: {p1_pass:.1f}% | Pass2 Val: {p2_pass:.1f}% | \"\n            f\"Consensus Kept: {b_agreed} quads ({cons_surv:.1f}% survival)\",\n            flush=True\n        )\n\n    print(f\"[Worker {args.worker_id}] [OK] Completed all {num_batches} batches successfully!\", flush=True)\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
print(f"✅ Staged worker process script at {worker_file}")

In [ ]:
# [Step 5/5] Master Orchestrator: Data Assembly & 2-GPU Parallel Execution
import os, sys, json, time, re, subprocess, threading
from pathlib import Path
from typing import Dict, List, Set
import pandas as pd
import torch

TARGET_REVIEWS = 5000
HALF_TARGET = TARGET_REVIEWS // 2
BATCH_SIZE = 100
MAX_TOKENS = 384
OUTPUT_DIR = Path("/kaggle/working/stage2_silver_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("🚀 ProductLens 3.0: High-Volume Silver Labeling Orchestrator")
print(f"   • Target Review Count: {TARGET_REVIEWS} (2500 Elec / 2500 App)")
print(f"   • Batch Size per GPU:  {BATCH_SIZE}")
print(f"   • Max Tokens:          {MAX_TOKENS}")
print(f"   • Output Directory:    {OUTPUT_DIR}")
print("=" * 72)

# -------------------------------------------------------------------------
# 1. Dataset Discovery & Assembly (strictly Electronics & Tech and Apparel & Fashion)
# -------------------------------------------------------------------------
print("\n📖 Discovering candidate reviews on Kaggle disk...")

EMBEDDED_SMOKE_REVIEWS = [{'review_id': 'smk_elec_0001', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Doesn't work on a Mac. It clearly says on line this will work on a Mac OS system. The disk comes and it does not, only Windows . Do Not order this if you have a Mac!!!!!!!", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0001', 'product_id': 'PROD_APP_SMOKE', 'text': 'sizes recomended in the size chart are not real. sizes are much smaller than what is recomended in the chart. I tried to put it and sheer it!. I guess you should not buy this item in the internet..it is better to go to the store and check it', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0002', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Great Graphics!. 2006 Coast to Coast for PC is great! The graphics are awesome... The game is what I had hoped for and a lot more!!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0002', 'product_id': 'PROD_APP_SMOKE', 'text': "comfy,warm.. but run big. just like the title says. Warm, comfy but run a size too big. I don't wear them out as they are just for lounging, so it's not that big of a deal for me.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0003', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Plug it in and it works!. Pull the battery saver tab, plug the unit into the wall and plug-in the device you want to control and everything works! I tried it from 25 feet away and through one wall and the remote turned my device on and off without a hitch!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0003', 'product_id': 'PROD_APP_SMOKE', 'text': 'Too Uncomfortable and Too Big. These pants were way too big (looked about 2 sizes larger), and they were incredibly stiff. They would have been very uncomfortable for my daughter to wear all day.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0004', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'nifty. It is very convenient to have the charging cable available without having the cord all over the place in the way. Works well too.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0004', 'product_id': 'PROD_APP_SMOKE', 'text': "Doesn't Fit. I ordered this DashMat for a specific vehicle, it was not the correct mat. I returned the original one and they sent me an exact duplicate of the wrong mat. I can't seem to get the correct one and therefore it sits in a box!!!!!!", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0005', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'This is Disgusting. The German Pinscher is a wonderful Family pet and this type of comments makes them sound dangerous. I own 3 German Pinschers and they are wonderful pets. I resent this type of product.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0005', 'product_id': 'PROD_APP_SMOKE', 'text': 'slip on 2 1/2 in. heel shoes. i am completely satisfied in the manner my order was delivered.the shoes are just as awesome as the advertisement showed them to be.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0006', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'needs $$ upgrade. Only has limited access to OBDII data stream unless you pay more money to upgrade the software.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0006', 'product_id': 'PROD_APP_SMOKE', 'text': 'Too Small. I have bought dockers for years but these are cut smaller than they usually are. Even with the "stretch" waist band they are tight. So much for an "extra" inch. Beware. Try these in a store before you buy.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0007', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'DO NOT BUY FOR G4 12" POWERBOOK. In a complaint to Macally regarding intermittent charging, their response was:Macally PS-AC4 AC Power Adapter for Apple G4Thank you for contacting us.Our PS-AC4 has problem with the 12" PowerBook G4. At this moment, we do not have any working one for your computer.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0007', 'product_id': 'PROD_APP_SMOKE', 'text': 'Cut smaller than usual. I agree, these are cut small... Buy an inch or two larger than you normally do. The waist band does NOT stretch.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0008', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Consider not buying this product. Bought this guy for a Mac PowerBook G4 Titanium (the description clearly states this power supply works for the PowerBook G4). Never worked. Couldn't say if this is a fluke or if they are all of comparable quality.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0008', 'product_id': 'PROD_APP_SMOKE', 'text': 'Too bad..... Did not like the pants at all, and threw them in the trash rather than pay return postage. Flimsy, smelled of dye, cheap fabric, etc. Too bad.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0009', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Just what that old computer needed. This adapter was just what we needed to put new life into our old Mac. The power now stays on to allow for hours of use. It works great! Thanks!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0009', 'product_id': 'PROD_APP_SMOKE', 'text': 'False advertising. If you search for Super Saver shipping this advertisement comes up, but this product is not shipped by Amazon is does not go out with the Super Saver shipping, and the cost of shipping is nearly 20% the cost of the pants.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0010', 'product_id': 'PROD_ELEC_SMOKE', 'text': "junk. As others have experienced, mine starts a a charge/battery/charge cycle that is maddening, and probably no good at all for the battery. Don't buy.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0010', 'product_id': 'PROD_APP_SMOKE', 'text': "Wasn't what I expected. I ordered the item according to the chart that was associated with the size and it ended up being to big. The item is also like it is made from rubber, but because I needed it for this weekend I don't know what I am going to do at this point.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0011', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'It stopped working after a few weeks. It worked well for a few weeks, but then the battery of my iBook G4 stopped being re-charged.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0011', 'product_id': 'PROD_APP_SMOKE', 'text': 'Great product; great seller!. I got this item within a few days of my purchase. It is exactly what I wanted -- I look a whole size smaller, instantly. I tried on the Squeem in a boutique and fell in love, but they were asking almost $200 for it! I highly recommend this item and the seller.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0012', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'The adapter shown is not the one that was shipped.. I received a different adapter so I cannot really tell you about the Macally adapter. I received an adapter from Inspire and it does not have the longer cord, the velcro strap or the led lights. But it does charge my computer and it arrived quickly.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0012', 'product_id': 'PROD_APP_SMOKE', 'text': "ahi. I am 116 lbs. but after 4 kids (a set of twins) the tummy was not the same. squeem gives me a beautiful shape under my clothes...never look better. the only downfall it does not cut inches off or make you loose any body fat..after its off all the bulge comes out. It makes you look great thou...don't leave home without it.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0013', 'product_id': 'PROD_ELEC_SMOKE', 'text': "good cheap alternative. The only downside to this adapter vs. apple's is the lack of the light ring which tells you whether the computer is fully charged or not. Otherwise it's a great deal!", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0013', 'product_id': 'PROD_APP_SMOKE', 'text': 'Squeem. I ordered according to the size chart for my measurements and it was way to small!! Very uncomfortable.I returned it,', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0014', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Defective twice. I used one Macally charger for my G4 for a month before it died and stopped charging my computer. Amazon replaced the product and my second one lasted one day. I returned it and am now using an apple charger.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0014', 'product_id': 'PROD_APP_SMOKE', 'text': 'Sizing is Ridiculous. I measure as a Small and the Small and Medium did NOT fit. Update: I ordered a Large and wore it for 2 days and it stretched too much. I am not sure this will help, but I have worn the item so I cannot return the Large. I am not any smaller than when I began so the Large did stretch and did not shrink my waist and/or hips.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0015', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Save the money!. I got my second one of these recently. Now I keep a charger at work and another at home so I don't have to carry them back and forth. The one at home has been in use for a year now with no problems; they're more durable than the Apple ones at a fraction of the cost.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0015', 'product_id': 'PROD_APP_SMOKE', 'text': 'works great. Bought it 3 months after pregnancy with twins. I had gone down to my pre-pregnancy weight but my tummy was still not firm. Used the belt and reduced the waist to pre-pregnancy size after wearing it 8 times.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0016', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Apple G4 Power Cord. I just bought my second one. We have 3 Macs at our house and the cords do not hold up. I dreaded replacing them but I like my computer. I tried this the last time my cord went and it worked great, so when the most recent power cord fell apart I did not hesitate. It does the same job at a third of the price!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0016', 'product_id': 'PROD_APP_SMOKE', 'text': "Pass. The Sizes run really small, It flips up even though, they said it would not and the hooks don't hold. I am very dissapointed.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0017', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'lemon!. just got this charger yesterday to replace my broken apple charger.All this charger does is allow me to turn on the computer (sometimes), it does NOT charger my powerbook g4.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0017', 'product_id': 'PROD_APP_SMOKE', 'text': "Would be great but.... The package says it goes from lower belly to just under bust. I am a short woman and it doesn't reach my ribs leaving me with two huge rolls of fat on the front and back. It takes me in great, love that about it but because it isn't long enough I can't wear it anywhere:(", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0018', 'product_id': 'PROD_ELEC_SMOKE', 'text': "When it worked it was great. I had this charger for over two years. Last week while it was plugged into my computer, it just died. No blue light no current. I ended up splicing the connector to my old frayed Apple charger. The connector is much better than Apple's.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0018', 'product_id': 'PROD_APP_SMOKE', 'text': 'Wasnt worth. I bought a pink rubber squeem which had a seperate cotton cloth . That worked well, but i wonder why i dint buy the same and ended up with this. This product hurts me on my left hip and i got scars on my left hip.. :(', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0019', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Charges. This charger works great and charges as well as the apple product that costs twice as much. I miss the green and orange light at the end of the plug, but this one does have a blue light indicating it is plugged in.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0019', 'product_id': 'PROD_APP_SMOKE', 'text': "i love it... i realy love my shape wear, i got one for myself and one for my daughter and she's notice a big difference,", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0020', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Great singers some not associated with Mozart. Good sound. I recall this recording on LP in the early fifties. It conveys the humour and genius of Mozart with great singers although other than Steber they were not previously identified with WM.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0020', 'product_id': 'PROD_APP_SMOKE', 'text': 'Squeem. Love this product. I wear it when I am doing my cardio to help with weight loss. It is true to size.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0021', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Frida's more attractive soundtrack sister. Utterly amazing and unique sound. Her voice is so powerful.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0021', 'product_id': 'PROD_APP_SMOKE', 'text': "So Far So Good!. I've only worn this for a day so far, but I love it! I still have a little bit of a tummy from the birth of my son 8 months ago. This really holds you in, its comfortable and I don't think it shows through your clothes! I highly recommend!!", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0022', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Ninnia. This monitor is great. The service I got when I order was fast and great. I would order again.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0022', 'product_id': 'PROD_APP_SMOKE', 'text': "Squeeeeeze into your clothes. I really liked the Squeem. I'm a petite person, but have some love handles that I need to hide when I'm wearing tight clothes. The only problem is that it has boning which you feel in your ribs when it's tight.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0023', 'product_id': 'PROD_ELEC_SMOKE', 'text': "I love the young Jerry.. Jerry has such a humble approach in his character it is very easy to sink in. There is never a dull moment when he comes onto screen. I love his B&W's.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0023', 'product_id': 'PROD_APP_SMOKE', 'text': "does not fit me!. So according to the size chart on the box size small is for waistline 28-31my waist is 28 so i got the smallit does not even come close to being able to close and clasp!maybe it's just me but i am terribly disappointed", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0024', 'product_id': 'PROD_ELEC_SMOKE', 'text': "camcorder cable. This is a must have cable if you're going to transfer your videos to your computer. Sony does not supply the cable with its HD camcorders. If you edit your videos and/or make DVDs, you will need this cable.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0024', 'product_id': 'PROD_APP_SMOKE', 'text': "Caused damage to my stomach muscles!. Felt great until it dug into my waist while sitting causing damage to my surgical site. Now I'm in pain with and without it on.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0025', 'product_id': 'PROD_ELEC_SMOKE', 'text': "I got the same thing from CompUSA $3.99. I was shocked at how expensive this is so I searched other sites for one. I found a 4-pin to 6-pin firewire at CompUSA's website for $3.99 TOTAL (free shipping!) This Sony cable works great, but I can't tell the difference between it and my cheap one...thus the one star.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0025', 'product_id': 'PROD_APP_SMOKE', 'text': 'Great product. Just what I needed to get me headed back toward my pre-pregnancy waist. Bought an XL and probably could have fit into the large. Looking to down size soon!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0026', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Undecided - Be cautious. It worked for about a month. Even when it did work it was a hit or miss thing. I had to fool with it for a while. Maybe it was because I bought the cheapest one available. Maybe it was a knock-off, who knows? I have yet to try it on another computer to rule out it not being the cord. Fair warning.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0026', 'product_id': 'PROD_APP_SMOKE', 'text': "Not what I expected. This product was not what I expected so I returned it. Too tight, sizing was misleading. I'll stick with my Spanx.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0027', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Eats tapes. After about a year of use, ZR 40 started eating tapes. Do a google search on the topic, there are hundreds of people with the same problem. 'Remove Cassette' and 'Eject' (use those as keywords combined with 'Canon ZR' to view message boards postings) will appear on the screen, and that's the end of the toy.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0027', 'product_id': 'PROD_APP_SMOKE', 'text': "Too small. Usually a size XS to S but this girdle's small did not fit true to size. Had to return back to Amazon.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0028', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Not HAPPY. I purchased this for my sony camcorder. I have windows XP on my laptop and this cable does not work. My laptop does not even detect my camcorder when attached.............Its a waste according to me', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0028', 'product_id': 'PROD_APP_SMOKE', 'text': 'AMAZING. I waited a while before I reviewed so that I could give a GOOD review and this works amazing, trust me you need to get used to it but you WILL and please buy according to size chart it does stretch after sometime but you need to SUCK IT IN when you first put it in my lower belly pudge is almost gone!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0029', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Memoir or Memory?. Although Derrida can be difficult to follow sometimes, this work is extremely intriguing and thought provoking. Well illustrated as well, as should be expected considering it was the catalogue for an exhibition.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0029', 'product_id': 'PROD_APP_SMOKE', 'text': "Its amazing!!!. It really works! I am in love. I was trying to get my pre-pregnancy 21 in waist back for 10 months now. I got to 23in and kind of got stuck there. With Squeem I went from 23 in waist to 21in in just 2 minutes! Start hooking it from the top, much easier. Buy it! You won't regret!", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0030', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Charger works great. The charger works fine as advertised. The location of the of the charger inlet on the phone itself is not very good - makes it awkward if you are charging and trying to talk.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0030', 'product_id': 'PROD_APP_SMOKE', 'text': 'LOVE THIS!. This thing is freakin awesome! You can stand it up, lay it down, or rearrange to your liking. They are sturdy enough to hold several things: shoes, baby bags, blanky, and more! I use them as cubbies for my kids so they can stuff the things that we take with us everyday in there. I love it, and I am planning on getting more.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0031', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'dont know!. the program looks fun but the software i recieved was corupt it wont load on any machine.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0031', 'product_id': 'PROD_APP_SMOKE', 'text': 'Perfect gift for the family. Purchased iwth Amazon Prime, it got delivered in two days without any damage. Shape and size are perfect for my sister who is getting this for Xmas and to use during her house parties.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0032', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Manhattan Transfer can't sing!. I can't understand how Manhattan Transfer gets all those studio musicians to play on their albums. I mean, it just makes their singing sound worse against the best drumming (Steve Gadd), bass playing and sax playing in the world. The two stars are for Kafka and the studio musicians only.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0032', 'product_id': 'PROD_APP_SMOKE', 'text': 'small. The shoes would have been awesome but..... the sizes run extremely small.I even ordered a size larger than normal thinking maybe this would happen.The shoes are still too small..', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0033', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Frightening, relevant and persuasive. I was most struck with the information about what happens to us physically when we watch television. The very act of watching itself, without reference to content, changes the way our brains operate. I recommend this to all teachers, parents, and broadcast journalists.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0033', 'product_id': 'PROD_APP_SMOKE', 'text': 'Cheap material. This tool is made of cheap material and never work or extract tripped screw. I have tried so many times and eventually have to buy different brand tool to make my work finished', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0034', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Suck it Michael Bay. Boobies, explosions, and maybe the fight/transformation scenes would have been awesome with theater version basss/surround sound. A waste of time. Unless you're drunk. Then it's ok.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0034', 'product_id': 'PROD_APP_SMOKE', 'text': 'Not true size. Went by the sizing chart and still did not fit right. Plus the way they explained the sizes really confused me. So maybe it was my fault, but I will be purchasing from someone who has the actual size I can understand.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0035', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Disappointing!. Just got this Blu Ray and just found out there is no DTS sound unlike what the description says on Amazon. Wait for the better version!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0035', 'product_id': 'PROD_APP_SMOKE', 'text': 'BBW. I loved the feel of the hose unfortunately the cotton lace top has no give and therefore is very uncomfortable and cuts into your thigh.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0036', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Awesome. This Blue-ray has great picture and sound, plays awesome on my 46" LED LCD. I couldn\'t ask for anything more.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0036', 'product_id': 'PROD_APP_SMOKE', 'text': 'Not great. I guess these would be good if you had foot surgery and wanted to place something lose and protective over bandaging, but not for under a boot cast. I was under the impression these would be tall enough to almost reach my knee. They only go about 4" above the ankle. Very wimpy, probably wouldn\'t hold up to too many washes.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0037', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'no more service. eMarker service will be end by the end of September.Refund service has already started.I just got the device last month and am ready to refund now.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0037', 'product_id': 'PROD_APP_SMOKE', 'text': 'Batgirl was a Christmas hit!. My 3 year-old niece loves to dress up, so we got her this costume for Christmas. She loved it and spent the rest of Christmas vacation wearing it.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0038', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Product returned but replacement excellent. The product was damaged when received so I returned it. Received a phone call asking the problem and had a replacement within a few days. Appreciate the response.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0038', 'product_id': 'PROD_APP_SMOKE', 'text': 'poor. Was missing the belt for one thing was to small had to order another one just so she could use it they need to put the correct size on the item we ordered a 5-6 got ot and the item said fits size 4-5', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0039', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Samsung 8mm video camera. The on/off switch is stuck. I cannot therefore record or review tapes. I pried the switch and made a temporary contact and confirmed that the camera can record and play-back I would like to be able to return it for full credit or have it repaired.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0039', 'product_id': 'PROD_APP_SMOKE', 'text': "No belt!!!. My costume was a small and didn't fit my 5 year old wayyyy to tight and it dd t come with the belt!!!", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0040', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Unable to install. My title says it all.I am running Windows XP with all service packs. AMD Athlon 64X2,4 GB Ram 2 hard drives, tons of free space.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0040', 'product_id': 'PROD_APP_SMOKE', 'text': "Nice Basic Mug. I like the size and style of these mugs, they go with anything and look like they'll last a long time.--- EDITED REVIEW --- Long term? not very durable. All but one have broken.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0041', 'product_id': 'PROD_ELEC_SMOKE', 'text': "NEVER BUY THIS PRINTER!!. I spent over 2 1/2 hrs. (after 1 hr. myself) on the phone with HP trying to install this printer. It never would work right. The HP tech said there is a problem in the software program of this printer. This HP DESKJET 940C is everyone's worst installation nightmare!", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0041', 'product_id': 'PROD_APP_SMOKE', 'text': "size of mugs. The mugs are very nice, but I guess I didn't think of the 16 oz. size, as a result they are much larger than I would have liked them...other than that they are very nice and will be put to use...", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0042', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'It worked fine until I bought a new computer. I owned this printer for more than a year. It worked fine until I bought a new computer, I unplugged it, moved to a new place, and replugged it. The power supply burnt out. The printer was dead. I wasted 2 packs of unused expensive ink cartridges.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0042', 'product_id': 'PROD_APP_SMOKE', 'text': "Bratz Helmet. When I ordered this product, I didn't see a size chart and I was sent a size that is not suitable for the purpose intended.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0043', 'product_id': 'PROD_ELEC_SMOKE', 'text': "three's company. this is one of the best comedy ever in the world tv until now i am still watching this comedy every time", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0043', 'product_id': 'PROD_APP_SMOKE', 'text': 'bloody awful. sounds like a third world elementary school class on a field trip to a crack factory. if this music is so revolutionary and worldly, how come the only people listening to it are skinny white boys who wear hats with earflaps indoors?', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0044', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'classic television. What can I say about this show except it is a good example of what a classic series is all about. Any John Ritter fan has to add this to their collection. I intend to purchase the entire series.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0044', 'product_id': 'PROD_APP_SMOKE', 'text': "Wolf in Sheep's Clothing!!!!!. I'm thinking of cancelling my subscription. Ever since Time Warner Publishing acquired Essence, it seems to be cantering to dark-skinned white women and NOT Black women.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0045', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Solid Prequel to Superior "Warrior". Finally this thing has been enhanced for surround. It is about GD time!!! Lacks one star because Road Warrior takes it one step to the next level of high intensity. Interceptor motor kicks butt on the old surround sound rear channels too!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0045', 'product_id': 'PROD_APP_SMOKE', 'text': 'Love it!. I am very happy with the weight and quality of this product. I wear this necklace often and love it!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0046', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Doesn't work in anything I put it in.. I have nothing good to say about Kingston memory. I tried installing this module in an E-machines T1220 1.2 GHz Celeron system that is only 3 months old and no go. Wouldn't work in two other systems I tried. The same spec memory from Centon and PNY both worked without any problems. Mine is going back.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0046', 'product_id': 'PROD_APP_SMOKE', 'text': 'Plasplug is JUNK!!. By something else. This is the worst piece of machinery I have ever bought. Virtually impossible to cut any size tile on the diagonal. Typical imported junk!!!!!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0047', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Help!. Hello everybody! I ordered this product but the first one that i received didn't work, and now i have the second one and it still doesn't work. It might sound weird, but i put batteries in it and it doesn't start working. What should i do? Thanks!", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0047', 'product_id': 'PROD_APP_SMOKE', 'text': "Very Low Quality Binding/Paper. This edition is basically newsprint with a hard binding. What a racket! Signet Classics (Pocket size) with introduction by Marcelle Clements.Tess of the D'urbervillesAnd the binding is not good, either--don't know what the word for it is but no left margin.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0048', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'PLEASE DO NOT PURCHASE. WASTE OF MONEY. FIRST TIME BUYING SUCH AND IT NEVER WORKED. MOTOR NEVER TURNED WHEN BATTERY WAS PUT IN. SUCH BLATANT ROBBERY!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0048', 'product_id': 'PROD_APP_SMOKE', 'text': 'Nice color. I looked far and wide for a medium to light brown eyeliner pencil and this one turned out to be perfect. If you have blonde hair, you can look a little trashy wearing black eyeliner, so this is a wonderful find.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0049', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'disappointed. I never actually got to use this product, when it arrived the bottom of the battery pack thingy was missing and when I attempted contacted the California Exotics customer service via both email and phone call I never received a response.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0049', 'product_id': 'PROD_APP_SMOKE', 'text': "Men's High Tec Boot. This boot is comfortable and gives me the ankle support I've needed for any type of hiking. I look forward to many years of use from these boots.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0050', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'dont bother with XP. I Have had trouble with HP and XP since upgrading. Although they say it is xp compatable with driver, it is not. Will cause system to lock up and when I called HP they blamed XP. Bought a Epson and first time install without a hitch. Hope this helps and God bless you!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0050', 'product_id': 'PROD_APP_SMOKE', 'text': 'Good Boots. Comfotable fit on the loose side. I use a heavier sock and works just fine. Recommended and would buy again.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0051', 'product_id': 'PROD_ELEC_SMOKE', 'text': "might be a good scanner. my scanner hasn't worked since the day i took it out of the box, installed the software, and the green light is on but when i hit scan it does nothing! looks like it's a dud. lol.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0051', 'product_id': 'PROD_APP_SMOKE', 'text': 'good boots. i usally buy these boots from famous footwear and pay like 80 to 90 bucks so i found them on amazon alot cheaper, thanks', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0052', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Very Frustrating Piece of Office Equipment. I have had the displeasure of working with this machine for over 4 years. Its incompetent feeder is only matched by the uselessness of the software it comes with. Save yourself the frustration and buy another scanner.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0052', 'product_id': 'PROD_APP_SMOKE', 'text': 'Right Boot Defective. The right boot sole was defective, had like a ridge on the outer edge making boot very uncomfortable to walk in. Return and refund procedure was painless. This was the second pair of these boots that had defective sole.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0053', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'breaks easily and has bad warrenty. the warrenty on this device is only 3 months and it breaks quite quickly. the cover kept popping off and when it broke creative said the warrenty ran out', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0053', 'product_id': 'PROD_APP_SMOKE', 'text': 'Hi-Tec Boots Altitude IV. This is my 2nd pair. This pair was stiff and needed some break in, but feel great now.Would recommend', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0054', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Ericsson Vehicle Hands-Free Solution HF 7600. I have bought 2 of these items for my 2 cars, but it burns the batteries every time! It even burns the phone, sometimes.Ranieri', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0054', 'product_id': 'PROD_APP_SMOKE', 'text': 'Great Boot. These Hi-Tec Boots are excellent and very good quality.I use them while I am a rural carrier. they are a great work boot.I recomend to all. Peter.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0055', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Quit working after 3 hours. The adapter quit working after 3 hours of use. I returned it and got a full refund. The customer service for the return and refund was excellent, and I would not hesitate to deal with this company again. But I won't buy any replacement power adapter for a 4-year-old laptop again. I got a new laptop.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0055', 'product_id': 'PROD_APP_SMOKE', 'text': 'not good. 3 months with them and my left boot it open in half i think the glue they use is not good i mean 70 bucks for 3 months is not worth it', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0056', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Works Great. The biggest minus is having to remove the phone from the cradle and unplug the cord from the phone to remove from the car. Other than that minor inconvienence it works great', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0056', 'product_id': 'PROD_APP_SMOKE', 'text': "VERY NARROW. This pair of boots are too tight and between sizes is big jump,i don't know if there is any other way I can stretch them a bit,they are also only made for soft soil , asphalt concrete will finish them too fast", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0057', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Jabra Eargels Great Help for my Jawbone II. While I loved the sound of my Jawbone II headset, it would not remain secured with the eargels provided. These Jabra eargels work wonderfully and keep the headset comfortably and securely in my ear.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0057', 'product_id': 'PROD_APP_SMOKE', 'text': 'hiking boot. We bought these for my husband. He has a rough time with shoes of any kind. These needed some time to be broke in but beyond that are very good. Good price - good value for the money, we think.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0058', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Jabra Ear Gels. I ordered the ear gels to go with my Jawbone Bluetooth headset. The problem that I see is that the ear gels are too bulky when combined with the Jawbone.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0058', 'product_id': 'PROD_APP_SMOKE', 'text': 'Not As durable as i hoped. Lasted about as long as a pair of boots from wally world for half the price. I live a on a small farm and I do some welding and construction. I guess for a city slicker they will last a life time.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0059', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Don't Bother. Don't bother. It is next to impossible to get this product to work. Tried everything (USB, download driver, etc.) and nothing worked - with XP Pro. ended up in the trash.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0059', 'product_id': 'PROD_APP_SMOKE', 'text': 'Boots. I have an old pair and they are great in the winter and comfortable. SoooooI ordered a second pair for this winter and by the looks at the snow fall yesterday, it was a good choice.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0060', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Hard to trust any other battery. I got an off-brand battery as an extra for my SD800 IS and was disapointed with the life. The Canon brand is worth the extra cost.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0060', 'product_id': 'PROD_APP_SMOKE', 'text': "Great item!. I bought these as a replacement when our new puppy chewed the end off of a similar type lace on my husband's favorite shoes. They come with easy to follow instructions and after a few minutes all was as good as new and harmony was restored! (shoes are now kept well away from puppy!)", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0061', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Perfect. I bought this to go with my SD700IS because I wanted to keep it all oem and just as the one that came with the camera I have to admit I have never seen a battery last longer I have had the camera for almost 4 months now and between the 2 batteries I have only charged both 1 time and believe me I use my camera allot!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0061', 'product_id': 'PROD_APP_SMOKE', 'text': 'Difficult to attach to boots. Too difficult to attach to my shoes...I prefer to tie my laces, even though I may have to do it more than once.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0062', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Battery. great for when you need an extra battery. It is a great deal on amazon because the cost would be more at a retail store.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0062', 'product_id': 'PROD_APP_SMOKE', 'text': "Very pretty and good quality. My two year absolutely loves puzzles and this has been a great addition to our collection.. It's sturdy and I don't think she'll be able to wear it out or break it.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0063', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'New battery in retail package. We wanted a spare camera battery for our Cannon SD870IS. It arrived on-time, and worked as expected. We no longer have to worry about our battery going dead on vacation and missing great pictures, we now have a spare.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0063', 'product_id': 'PROD_APP_SMOKE', 'text': "Didn't Last. I bought this for my 8 year old to sleep on in temporary housing during a corporate move. This bed didn't last two days and it came apart in the seam where we couldn't patch it. I would not waste my money on this product again.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0064', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Fine battery. I purchased this as an extra battery for a Canon sd800is. It is the same as the original battery and is Canon "branded". A third party battery is available locally at TWICE the price, and the Canon version for almost three times the price. This is a good buy.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0064', 'product_id': 'PROD_APP_SMOKE', 'text': '3 weeks and heard a big boom!. Heard a huge boom and I noticed the seam had given out and then this huge bubble formed in middle of the bed and then it ripped a few days later. I purchased the product on or around 10-6-2011 and before 11-1-11 it was deflated!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0065', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Perfect backup battery. Although it doesn't get use unless we're out long enough to need a second battery, this product is working just like the original and seems to be a completely legit cannon production (despite a few other reviews).", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0065', 'product_id': 'PROD_APP_SMOKE', 'text': "Lasted one month of everyday use. I used this product for one month of every night sleeping before it sprouted a leak in one of the seams on top that couldn't be patched due to the fabric. It makes patches virtually impossible to stay stuck.There you have it.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0066', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Knock off battery does not work at all. I ordered this battery, thinking that it was a \'real battery\'. Did not charge or work at all - says "Canon Made in China". If this is not a manufacturer\'s battery then I paid way too much for it. Oh, since it doesn\'t work at all I paid way too much for it.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0066', 'product_id': 'PROD_APP_SMOKE', 'text': "Cheap. This was a gift, but the fabric was cheap and didn't look great on or last well. However It was fun, and I got compliments on it all night.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0067', 'product_id': 'PROD_ELEC_SMOKE', 'text': "You get what you pay for. You get what you pay for. I have used these batteries as back up for the original Canon one. After 10 months with only occasional use, battery won''t take or hold a charge. Do you self a favor, and buy a Canon battery. You'll save money in the long run.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0067', 'product_id': 'PROD_APP_SMOKE', 'text': 'waste of money. the kit doesnt even fit the heater outlet vent on the 75K btu model. half the parts you wont need, just go to a hardware store & buy the ones you need for half the price.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0068', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Brand Name Battery. A no-brainer! The best battery for your Canon camera. What else would Canon want in their cameras? DUH!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0068', 'product_id': 'PROD_APP_SMOKE', 'text': 'Does not work with PowerMac G5. Although the specifications state compatibility with macs G3 or higher, this card will not fit into the pci slot of a new G5.(...)', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0069', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'i have get my order. I never got the battery and im still waiting on it, I want to know whats going to happen cause im still waiting', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0069', 'product_id': 'PROD_APP_SMOKE', 'text': "Junk,Don't wast your money.. I got these in hope It would give me a good fit to my jawbone 2, but they did not work for me. I think they even made it less of a fit.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0070', 'product_id': 'PROD_ELEC_SMOKE', 'text': "JUNK!!!!!!!!!!!!!!!!!!!!!!. wow I had the exact same things happen to the one I got for my kid bad motion sensor and bad display. he wanted it so much we tried returning it twice after the third one didn't work even my six year old said forget it. I wish i could have given it a black hole instead of 1 star. JUNK!!!!!!!!", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0070', 'product_id': 'PROD_APP_SMOKE', 'text': "Makes a Jawbone 2 better. I hated the fit of my Jawbone 2.0. It would never stay in no matter which ear loop or ear bud I used. Someone recommended these to me, so I thought I'd give them a try. After 2 months of using it, I'm pissed I didn't get them sooner. Before chucking your Jawbone 2.0, try these.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0071', 'product_id': 'PROD_ELEC_SMOKE', 'text': "This toy is the WORST!. This toy is the worst! You can't even get past the Password screen to play anything. I got tricked that it was a good toy, but it's not. And I don't want you to get tricked, either. I thought it was a good toy by looking at it in the wrapper, but it wasn't.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0071', 'product_id': 'PROD_APP_SMOKE', 'text': 'great product. people should buy headsets to fit these things bc they are so essential..A must have for any headset that will fit them', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0072', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'MY SOON TO BE NEW TOY. I Cant wait till christmas then i will get my advace pet i seen what it could do so i cant wait.hey does it come with a jack in cord so u could like jack in to ur computer and add stuff.If so that joint is cool right protoman (yep)!!! hheehehehe im 14 years old and i approve this toy to be the best.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0072', 'product_id': 'PROD_APP_SMOKE', 'text': "Ear Gels. None of them actually fit my ear or the other person I brought them for. They were way too big. If you have small ears don't purchase this item.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0073', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'NIkon EG-D2 Audio Video Cable for Nikon. This item is very well built, and works great with the Nikon Camera it was purchased to support. I recommend this item.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0073', 'product_id': 'PROD_APP_SMOKE', 'text': "Waste of money. This product ended up being a waste of money for me. They didn't fit too well so I ended up just using the original Jawbone ear piece with the these.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0074', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Defective and more?. I had 2 problems with the product. First the sound would cut out of one ear unless I played with the connection and second, using the extension caused the sound in my left and right headset piece to reverse (right sound came out of the left ear and vice versa). I have requested a refund and will not reorder if this is normal.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0074', 'product_id': 'PROD_APP_SMOKE', 'text': "don't buy. these were advertised to fit the habra stone. they did not. i don't know if they changed the advertisement since then but it was aid to be a package with the habra stone so i got it. not even one piece fit. threw them away", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0075', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'It works with my seven year old Sony Bravia. My husband is using the 3.5mm Stereo Headphone Extension to watch TV, with my seven year old Sony Bravia, while I listen to my audiobooks on my iPod. We have a very quiet home. He is happy with the sound.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0075', 'product_id': 'PROD_APP_SMOKE', 'text': "Liked the ear piece. I quess I ordered the wrong ones, did not fit my ear piece well. I've never been able to find any other pieces for this particular ear piece so I quess I just have to trash it.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0076', 'product_id': 'PROD_ELEC_SMOKE', 'text': "AWESOME. i bought for my dad so he doesn't have to sit 3 feet away from the TV while my mom is sleeping. Now he can sit on the bed and lay down with another 5 ft to spare..", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0076', 'product_id': 'PROD_APP_SMOKE', 'text': 'Does not fit Jawbone Era. The specs for this product state that it universally fits all Jawbone headsets. It does not. It does not fit the Jawbone Era.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0077', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'headphone extension.. almost a year of use, and it still works great, this runs from our tv, to my treadmill, I exercise 6 days a week, and this is a fantastic extension.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0077', 'product_id': 'PROD_APP_SMOKE', 'text': 'The Coolest. I wore this shirt to work one Friday (casual day) and one the my co-workers told me "I\'ve worked here 7 years and that is the coolest shirt I\'ve ever seen." Need I say more.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0078', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Works as desired. Not completely perfect, but reliable, and frankly, it extends the reach of my headphones. This is all I need it to do.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0078', 'product_id': 'PROD_APP_SMOKE', 'text': '89 chevy s10 single cab. i wanted to know if this will fit my 1989 chevy s10 single cab regular bed please let me know as soon as possible so i can place my order', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0079', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Good deal. I am am a musician so I use these with earbuds coming off my computer within my headphones which is powered from my Marshall Amplifier. They transfer the sound well and are very well made.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0079', 'product_id': 'PROD_APP_SMOKE', 'text': "You get what you pay for, and we didn't pay much :-(. The fit of the fitted sheet is LOUSY - the first night we had it on the mattress, it came off. I guess Amazon sold them cheap to get rid of them.DO NOT BUY THESE!!", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0080', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Stopped working after 2 months. Ok while it lasted. However this cord stopped working after two months. Save your money -- this is junk.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0080', 'product_id': 'PROD_APP_SMOKE', 'text': 'Great product. Looks and fits great! My son is in high school and these are a part of his uniform. He loves them....', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0081', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Sounds good. Sound quality is good for such an inexpensive item and long enough to let me move freely around in my office. Meets my needs', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0081', 'product_id': 'PROD_APP_SMOKE', 'text': "Quality Product. Pro - Product is of the highest quality. Con - I'm looking for rivets to fit this unit and can't find them!! Amazon do not carry them.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0082', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Didn't get what I ordered. I ordered the headphone extension (1 male end/1 female end) and got a 15 foot auxiliary plug (2 male ends). I decided that it would be cheaper to buy a stereo coupler of 2 female ends than it would be to ship it back.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0082', 'product_id': 'PROD_APP_SMOKE', 'text': 'Veclo stap to large. I like everything about these shoes except the velco stap is enormous and overpowers the entire front of the shoe. My son looks like he is wearing orthopedic shoes, I had to return them he just looked way to goofy', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0083', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'What a rip off!. I needed a charger because i misplaced mine. I an stunned they want $36 for this thing that should not cost more than $10.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0083', 'product_id': 'PROD_APP_SMOKE', 'text': 'THIS GAME STINKS!.... The only way to score on this game is dekes, rebounds and defelections. The gameplay stinks, the commontary stinks, DO NOT...waist your money on this game.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0084', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'inaccurate description. The description stated that the condition of the charger was like new. I got it in the mail after about 2 weeks (from Japan), and it had scratches all over it. I would not buy from this seller again.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0084', 'product_id': 'PROD_APP_SMOKE', 'text': "Great Socks. I wear these socks on a daily basis and I am very happy with them. They are about the only pair I've found that will actually stay up on/above my calves for any period of time.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0085', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Cheaper at Amazon than anywhere else. I didn't want to risk a generic charger and got the Canon brand at the lowest price at Amazon.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0085', 'product_id': 'PROD_APP_SMOKE', 'text': "Not as described. Package says one size fits most queen...well what the heck is most queen because a 1x and a 3x are too different sizes, I am a 1x and this didn't fit over my curves.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0086', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Battery Charger. Purchased this item to have a second charger on hand for my new camera. Identical to the OEM at a very good price.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0086', 'product_id': 'PROD_APP_SMOKE', 'text': 'Ultra Force Field Jacket. The real deal. Wore one when riding my motorcycle in Europe during the winter. Field jacket, liner, and undersweater. 28 degrees and still comfortable. Great investment.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0087', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Exactly Like My Original. I lost my original charger and bought this to replace it and it works just like the original. Very happy about this purchase!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0087', 'product_id': 'PROD_APP_SMOKE', 'text': 'Snapped right into my 03 Accord. This is a pretty good product. It fit nice and tight in the doors of my 2003 Accord without having to drill any holes. I gave it 4 stars instead of 5 because I had to do some customization to get a tight seal around the speakers (5 1/4 in).', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0088', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Cannon charger. This is a great product. I thought that I had lost the orignal(left in a hotel room)so I needed a replacement. It is the best! Found my other charger and now I have two. Great! I use the other one for the office. Great product from Cannon.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0088', 'product_id': 'PROD_APP_SMOKE', 'text': "doesn't work if you wear glasses. You look thru a pin-hole to get the sighting, but you can't see thru the pinhole if you wear glasses. If you take your glasses off then you can see thru, but if you're near-sighted like me then you won't be able to see very far in the distance to tell what you're sighting.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0089', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'nonsense. Can not get the pictures off the camera. Keeps saying faliure when try to download or go to website.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0089', 'product_id': 'PROD_APP_SMOKE', 'text': "Had to return it. Unfortunately I had to return the Blue Ant because I couldn't charge it without the cycle turning itself off. The size & fit were good; it just wouldn't take a charge.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0090', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'I bought this as a Christmas present!. Now my daughter will not stop aggravating me because we cannot download the pictures off of the dran thing and wants a new camera, so now, I have not only wasted my money, but I have to buy her a more expensive camera that will work that she will break or lose.Thanks so much Vivitar!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0090', 'product_id': 'PROD_APP_SMOKE', 'text': "Excellent. Reached in time for my daughter's school . My daughter enjoy it. It was for her course material so not much can be said.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0091', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Didn't work. Tried to play on two different Blu-ray players and it simply would not play. I could only get it to play on my computer. No good.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0091', 'product_id': 'PROD_APP_SMOKE', 'text': 'Ballet Flat. For it being a Clark Shoe, it had NO arch support. I returned item but had to pay the shipping.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0092', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Does not work with palm easily. Did not work with my Palm T|C right away.I've not used this for any other purpose. I expect either Palm or SD to release proper software to have it work with Palm T|C.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0092', 'product_id': 'PROD_APP_SMOKE', 'text': 'This was a present for my husband. My husband said that they shoe is very comfortable and he would buy them again. I thought the price was very reasonable', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0093', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Problems with drivers and Syncing - Don't Buy. Drivers for XP gave trouble. Had fewer problems with Win ME.However, this baby has problems syncing sound and video. Check all forums on the web, this is a problem everyone has.Doesn't look like ADS is about to fix it any time soon.A definite DON'T BUY.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0093', 'product_id': 'PROD_APP_SMOKE', 'text': 'could not get my dog to wear it. i tried everything but he hated it. it was a no go. even though i set it to a snug fit, he kept pawing at it until he was finally able to get it into his mouth and chew on it.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0094', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'bad. Not happy with this product lots problems!!!Not work at all for me. Had a person who is computer nam look at it and everything. He not get it either.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0094', 'product_id': 'PROD_APP_SMOKE', 'text': 'It\'s a fake. The collar actually works fine but it\'s not a Halti. It\'s a \'Holt" Looks and works the same but it\'s not a Halti which is what I was ordering.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0095', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Glad I finally found it!. I remember watching this on TV and taped it them. Somewhere along the way I lost the tape and finally found it on here. It is a FANTASTIC show!!! A must have for anyone...no caveats, just anyone who likes music!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0095', 'product_id': 'PROD_APP_SMOKE', 'text': "Doesn't work with either of my dogs..... We have two Bernese Mountain dogs. Niether will use this collar. They fight to get it off the entire time. Tugging on it does not get their attention. PetCo also expects you to just mail items back with no documentation. Return for this is not handled by Amazon...", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0096', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Junk. This wireless camera only works 5 ft away from the receiver. It is supposed to be 100m. What is the point of wireless if you have to be that close?', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0096', 'product_id': 'PROD_APP_SMOKE', 'text': 'Works well. This Halti collar worked well at controlling my Great Dane. He learned not to pull very quickly.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0097', 'product_id': 'PROD_ELEC_SMOKE', 'text': "stan larano. wish it had ac cord,batteries on high only last a few hours no good for my purpose,won't go overnight.......good for a few hours outdoors.......Wenzel LED Fan Light", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0097', 'product_id': 'PROD_APP_SMOKE', 'text': "Sizing instructions are off. The Halti must be correctly sized - and unfortunately the sizing info to determine the correct size is poor. The Holt sizing chart is based on breeds - and didn't work for my mutt. Way too big. Why don't they provide the sizing info based on the circumference of the muzzle and neck?", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0098', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Razer charger. works amazing in my car without any problems. Lights up a bright blue when charging and actually chargers my phone up.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0098', 'product_id': 'PROD_APP_SMOKE', 'text': 'LOVE THIS WALKING COLLAR. This walking collar has the greatest design. I do not have to have great upper body strength to control my excited boxer. She walks next to me like she is supposed to. No more pulling me down the street. Bravo', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0099', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Counterfit item - but works OK. According to the Motorola guide, the charger I received is a counterfit: It has a illuminated blue dot rather than a fully illuminated "M". It works OK, but should not be represented as made by Motorola.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0099', 'product_id': 'PROD_APP_SMOKE', 'text': 'Pretty. I got this shirt when it was on sale for $10 so I snatched it up quickly. I like the shirt a lot. It is very pretty on. Order the size that you normally wear. I made the mistake of ordering a large instead of a Medium. The shirt is way to big. So maybe even order a size smaller because it is just a bigger shirt.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0100', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Would have been nice except all the bugs!. This game could have been a nice RPG game.However, the large number of bugs makes the game unbearable.I've seen many computer games, and this takes the cakefor the number of errors, glitches, freezes, bugs...Save your money.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0100', 'product_id': 'PROD_APP_SMOKE', 'text': "The ad didn't say what size this thing was. I ordered this wrench assuming it would fit my water heater drain plug. Since there was no choice of size, I assumed one size fits all. Well, unfortunately, one size does not fit all and this drain plug is too small for my drain plug.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0101', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'new headset. the quality of this is very good. only problem is that i wish the battery would last longer. lasts about 4 hours', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0101', 'product_id': 'PROD_APP_SMOKE', 'text': 'Did not fit my Coleman E3. This appears to be perfectly fine as long as it fits. I guess the Coleman E3 does not use a standard fitting so it does not fit.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0102', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Great product!. I have an older digital camera that takes a compactflash card, but the new one I want to get uses smartmedia. What to do? No problem, I got the new camera and this Flashlink reader so I can use both cards and both cameras!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0102', 'product_id': 'PROD_APP_SMOKE', 'text': 'Every paint was dried up. I purchased this a month ago for my daughter for Christmas. She just opened them to decorate the shoes this morning and every single paint except the light blue is completely dried up inside of the tube. She is disappointed and I am annoyed.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0103', 'product_id': 'PROD_ELEC_SMOKE', 'text': "disappointed. I've seen better episodes on TV. Forgettable, don't even remember what it was about and it wasn't long ago that I saw it.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0103', 'product_id': 'PROD_APP_SMOKE', 'text': "Won't fit a 2000 gsxr600!!!!!!!!!!!!!. The pipe for a 2000 GSXR600 has 3 bolts on the motor side, the pipe for 2001-2003 has 4 bolts, the holes don't line up. If you buy this pipe for a 2000 GSXR600 it WON'T FIT!!!!!!!", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0104', 'product_id': 'PROD_ELEC_SMOKE', 'text': "won't play on my xbox. when I insert the disk the screen says you must attach hard drive to play. i have a 4 gig hard drive attached but it still won't play", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0104', 'product_id': 'PROD_APP_SMOKE', 'text': "Bronze. I was not pleased, there were glue stains and rhinestones missing, but because I was pressed for time I had no choice but to keep them (they were to wear at my daughter's wedding.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0105', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Camera, yes - Camcorder, no. It works in my Sony DSC-S85 camera, but not in my Sony camcorder. It gives a format error. I'm not sure why the difference because it works fine in the camera. If I would have known that it would work only in the camera, I would have purchased the Sony Stick.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0105', 'product_id': 'PROD_APP_SMOKE', 'text': 'not easy to fit. i got a pair for my daughter. however, when she tried it on, we had a hard time fitting it because it was quite tight. though it fits her, it was not comfortable. she ended up not wearing it at all.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0106', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'great subsitute for sony memory stick. Great subsitute for sony memory stick. cost less, same quality', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0106', 'product_id': 'PROD_APP_SMOKE', 'text': "adidas Kids' Absolute TRX Firm Ground Soccer Shoe,White/Black/Gold. The price was perfect for a brand new cleat. It wasn't shipped as fast as another competitor and was delayed by 1 day. However, the price made up for the time of delivery.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0107', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Numbingly bad.. Only in this, the age of mediocrity, could a piece of rubbish like this get made. Owen Wilson turns in one of the worst performances in recent memory. This was made strictly for the idiots who find MTV worthwhile part of our culture.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0107', 'product_id': 'PROD_APP_SMOKE', 'text': "These diapers don't do their job.. These diapers have a design flaw. When my baby has a BM, a small amount seeps through the diaper so I put two of these on so her clothes do not get messy.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0108', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Won\'t work in Dimage 7i. Inserting this card in my new Minolta Dimage 7i (digital camera), I get an "Unable to use card" error. The card is evidently incompatible with the 7i. Others have had the same experience. See the digital camera forums.Check the compatibility chart at minoltausa.com before you buy!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0108', 'product_id': 'PROD_APP_SMOKE', 'text': "Crap.... I wish there had been SOMETHING in the product description that said these diapers were SCENTED! You'd think you'd have to claim that. Now, who knows if i'll be able to return them as i did open the package, only to find they are scented. my kids can't wear these!", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0109', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Very fast!!. Simpletech's 256MB CF is very fast. I use it with my Canon camera and love it. It gives me quick transfer rates with high resolution. Same time and money and any headaches.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0109', 'product_id': 'PROD_APP_SMOKE', 'text': "Awesome diapers. they fit my daughter perfectly and i dont have to worry about leaks or anything anymore, before i started using Pampers Baby Dry Diapers my daughter's diaper would always leak esp when she pooped but now with Pampers Baby Dry Diapers they can totally handle anything she dishes out.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0110', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Great mount for my 40 inch LCD TV. We love this corner mount. It is plenty big enough and stong enough for our 40 inch Sony LCD TV. It's just what we needed!", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0110', 'product_id': 'PROD_APP_SMOKE', 'text': 'Great Diapers. I love pampers diapers. I have two boys and have used pampers almost exclusively. They have fantastic leak protection and fit my boys better than other brands.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0111', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'No Widescreen. This is only avalible in full screen. Its lack of a proper aspect ratio. Disney is totally ignorant in this issue. When disney decides to release this in its in 1.85 anamorphic widescreen then I will buy it.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0111', 'product_id': 'PROD_APP_SMOKE', 'text': 'Quality issue. I purchased a box of Pampers baby dry size 1( 252 pcs) and found brown stained at the front of the diapers. Still have 2 unopened bags with brown stained.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0112', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'So far so good!. The scooter has a good looking, the bateries last up to 45 minutes of continuos use but, need to change fuses evry long ride or every charging process, so far so good!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0112', 'product_id': 'PROD_APP_SMOKE', 'text': 'cheapest price. I am always looking for the best prices on diapers. I like these overnight diapers - they work for us - we always use one size larger than our daily diapers and that seems to work.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0113', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Worked Great.. I had bought this some time ago, but never used it. Recently I had been having memory card errors on my Canon Camera. I decided to try the 4 in 1 cleaner and it solved my error problems', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0113', 'product_id': 'PROD_APP_SMOKE', 'text': 'BE AWARE. I purchased this package of 222 diapers size 3 and it is already 3rd time (from out of 10 changing) that our baby got whole his body covered with those jelly pieces from inside the diaper. Scary, i hope it is not toxic or something.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0114', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'I love this product. I have been searching the product like this for long time,I got one. The important feature are pronounciation and quickfind. Just by mouse over the word you can find out the meaning.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0114', 'product_id': 'PROD_APP_SMOKE', 'text': 'NARJIS. I used them just for a few days . i bought a big box and put them on craigslist for baby dry lovers. but i will not recommend any one to usethem. Every time i had to change layers of clothing with diaper. bad on laundry budget.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0115', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Pretentious. Found it stupid and annoying. This pretends to be a documentary made 5 million years into the future sent back to the present day, supported by bad computer graphic fantasy animals. In a word, pretentious.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0115', 'product_id': 'PROD_APP_SMOKE', 'text': "Leaking all the time. I have a boy and this diapers are really bad for him and for me :-/.They are leaking all the time, even if it is a new not over wet diaper.Buy something else if you don't want to wash the clothes many times a day !!!", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0116', 'product_id': 'PROD_ELEC_SMOKE', 'text': "My nephews love them!. I purchased 2 of them for my nephews so that they don't fight with one another. My only complain is the flashlight, very poor quality. The light keeps going off and on, even with a new battery. I think it's a connection problem. Aside from that... my nephews love them.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0116', 'product_id': 'PROD_APP_SMOKE', 'text': 'Pampers Reliable As Ever. I used Pampers for my 1st child and doing so again with my 2nd. They fit great on kids who are wider through the hips and behind.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0117', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'very poor quality. This item is on sale, for a reason. It is not any better than the original one that came with the speaker. Very thin wires. Very dissapointing. I had to go to best buy and spend $35 to get some better wires.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0117', 'product_id': 'PROD_APP_SMOKE', 'text': 'I love Amazon.. Pampers are great. Pampers from Amazon are better. I ordered in bulk and needed a larger size faster than I thought. Amazon picked up the unopened boxes from my house and credited my card. I then ordered the correct size. Great customer service.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0118', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Good Godowsky.. These are good performances, but I prefer those of Carlo Grante on Music & Arts. Recording sound also leaves something to be desired.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0118', 'product_id': 'PROD_APP_SMOKE', 'text': 'Horrible. Several of these diapers broke at the side seams causing the gel material to fall out and leave a trail everywhere our toddler went until we finally figured out where it was coming from. Happened with three more diapers in order and after vacuuming for over one our we gave up and tyre the rest out.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0119', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Works Good.. I Bought this card for a third computer in my house and its the ferthest from the router and works fine.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0119', 'product_id': 'PROD_APP_SMOKE', 'text': 'TERRIBLE! Leaks!. This is the worst nighttime diaper we have used. They leak 100% of the time leaving our baby drenched in her own urine! Awful. :( I highly recommend Huggies Overnites-zero problem with leaks and a great fit.We will not attempt to use these again.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0120', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Significant improvement over my USB wireless. I had a USB wireless adapter that got 50-60% signal strength. This this adapter, I get around 90%. I've absolutely no problems with this card and have been impressed by it's performance.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0120', 'product_id': 'PROD_APP_SMOKE', 'text': 'Great Diapers. This diaper is very comfy. My son really enjoyed wearing it. Hopefully the price can be a bit lower at the local retail store though.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0121', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'works well. does as advertised. Bought this because of current reviews and it\'s great. I have to "repair" the connection about once every 2 weeks on the machine being used with this card, but I never have to repair the connection on my other computer. Not sure why this is, but it\'s a really fast fix and never causes problems.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0121', 'product_id': 'PROD_APP_SMOKE', 'text': "Good for dogs too. I have 2 aging cocker spaniels whose bladder control isn't as reliable as it used to be. Size 6 fits well. We use these on them during the day if they're alone at home, and at night while sleeping. Keeps them from using the kitchen floor as a bathroom.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0122', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Works great. This product works really good. My computer is pretty far from any connection, yet it still connects great, and is usually very fast.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0122', 'product_id': 'PROD_APP_SMOKE', 'text': "diapers. I save from .04 to.07 per diaper (depending on sizes I have autoshipped and mr mom deductions) as opposed to walmart's price. This adds up to a minimum savings of $7.68 per box up to well over $10 per box on smaller sizes. I have 3 in diapers.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0123', 'product_id': 'PROD_ELEC_SMOKE', 'text': "After 3 years still no Vista driver that works. I'll never buy a HP product again - sitting here with a Laserjet 1020 I'll never be able to use again after upgrading to Vista. Tried all fixes and workarounds, no luck.HP LaserJet 1020 Printer (Q5911A#ABA)", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0123', 'product_id': 'PROD_APP_SMOKE', 'text': 'Diapers work great!. We use these diapers when we are out and about and cloth diapering is not an option. They work great and are an awesome value!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0124', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Dissatisfied with installation software. I was very dissatisfied that the HP LaserJet 1020 installation software did not provide acceptable printer drivers for Windows XP or Vista.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0124', 'product_id': 'PROD_APP_SMOKE', 'text': 'Good product.. I ordered the same size 2 months ago. It was good. Now, this is my 2nd time order. Good product. As expected!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0125', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Damaged after first use. The first time I plugged in this device, part of the plastic melted away and the device became useless. I consider it a total waste of money.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0125', 'product_id': 'PROD_APP_SMOKE', 'text': "Good product. As good as expected!! The only con is that it is supposed to fit babies from 17 to 25 pounds but my baby is only 18 pounds and it's already snug. Need to change to next size, other than that very good.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0126', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Works great. This was purchased for my mother-in-law and she loves it. Keeps her tea and hot coco warm for her as she sits in her recliner and reads or watches TV.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0126', 'product_id': 'PROD_APP_SMOKE', 'text': 'Diaper heaven. Not only do i love the pampers brand because it fits perfect on my 1yr old but also it holds all her "stuff" lol the amount is amazing i didnt have to buy diapers for over 2 months!!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0127', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Arrived defective. Bought three of them for use around the house to replace one that I have had for years.All three produced a strong odor of burning plastics and electronics.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0127', 'product_id': 'PROD_APP_SMOKE', 'text': 'Great diapers!. These diapers worked very well for my daughter, but I wish I would have bought the next size up because she is chubby and they are not that wide if your baby is closer to the maximum weight, but overall they work great.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0128', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Worked like a charm! FOR 5 MINUTES. Really wanted this to work, but sadly disappointed. Literally worked for about 5 minutes and then stopped heating altogether. How difficult can it be to make such a device that actually works?', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0128', 'product_id': 'PROD_APP_SMOKE', 'text': 'Love :). These worked great for my twins! super easy to manage with a squirmy infant! We had switched from cloth and tried a few other brands but these worked best.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0129', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'The heat is on all over the place ! ! !. Hey they work, I ordered one for the wife to use at the computer, now this is the third one and we have one at the jewelry table, and the living room ! ! !', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0129', 'product_id': 'PROD_APP_SMOKE', 'text': "Exploding diapers. Our daughter is 6 months old and 19 lbs. She has been wearing these size 3 Pampers, and we wake up to exploded diapers many mornings, with little gel beads all over DD. I'm switching to Huggies overnight, based on a co-worker's recommendation.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0130', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Safe?. I recently purchased this mug warmer to keep my coffee warm while working at my desk. I plugged it in, turned it on and there was a *pop* sound, a spark, and then a little smoke. Nice idea and all but I'll take the trek to my microwave over a fire hazard.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0130', 'product_id': 'PROD_APP_SMOKE', 'text': 'Baby Dry Runs Very Small. I moved my daughter up from a size 1 Pampers Sensitive to the size 2 Baby Dry. The diapers are the exact same size. We have a blow-out every BM. I am so mad right now. How can the sizing be so off? I am stuck with two big boxes now of diapers I cannot use.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0131', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'not happy. I find this cup warmer only keeps my coffee lukewarm at most even starting with hot coffee. I work at the computer and when I stop to reach for my cup, I want hot coffee, so it was definitely a disappointment.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0131', 'product_id': 'PROD_APP_SMOKE', 'text': "Runs small. I always get size 4 for my little guy, and so I ordered size 4 in Pampers Baby-Dry. I can barely fit them around his waist. My friend and I compared Pampers size 3 and 4 next to each other and we couldn't tell any size difference.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0132', 'product_id': 'PROD_ELEC_SMOKE', 'text': "1 time use only. After the first use, unplugging the cable. while it was cooling down cracked voice raised then it's not heating again.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0132', 'product_id': 'PROD_APP_SMOKE', 'text': 'Love this diaper. We have been purchasing this diaper ever since our daughter is 2 months old, we were using size 1, 2 and now 3. We do not wish to change to other brand. when our daughter was first born, we bought so many other brands of diaper, but only this one satisfy the need.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0133', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'gift. This was brought for my husband to finish his set. He has all of the seasons. It was one of our favorite shows on tv and it will be missed . at least this way we can watch it over and over .', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0133', 'product_id': 'PROD_APP_SMOKE', 'text': "Terrible diapers. I purchased the next size up for my son for nighttime. I have never had a leak this badly before where it went up his chest and sides. I'm going to try the Huggies nighttime diapers.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0134', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'EXCELLENT PRODUCT. This is an awesome product it charges super fast. I dont know what the other reviewer is talking about i can get 2 charges out of this even when i use my extended iPpod battery this is one product every person with an ipod should have.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0134', 'product_id': 'PROD_APP_SMOKE', 'text': "Very Thin and leaks. Tried pampers for the first time and it was size 5. It's very Thin and most of the time it leaks from the side when she poops.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0135', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Not good at all. I got one of these a year ago. The issue I had was that the thing only had about 15mins of power before shutting down. That is usb or powering a 7w fluorescent bulb AC.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0135', 'product_id': 'PROD_APP_SMOKE', 'text': 'Online Pampers are poor in Quality. Pampers that I bought online are very poor quality than that are purchased from stores.Pampers Baby Dry Diapers Economy Pack Plus Size 4 192 Count-Jhansi', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0136', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Instant Aversion. Bought version 2.0 of this at a local bricks & mortar retailer. I couldn't get the software on disc 1 to work. Exchanged the disc - tried it on a few different computers but no luck, so I'm shopping for something else now.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0136', 'product_id': 'PROD_APP_SMOKE', 'text': "works... that's what matters here. Pampers are our sole brand in this house. They fit the best and they hold the best. That's all the really matters to me.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0137', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Uh, well.... Really mediocre romantic comedy featuring a brown-haired Helen Hunt before Mad About You and As Good As It Gets, Kelly Preston before her career surge, and Andrew McCarthy on the downside of his Brat Pack run. Shows up occasionally on cable, and best left there.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0137', 'product_id': 'PROD_APP_SMOKE', 'text': "big diapers. Now I am sitting on a big box of diapers that don't even fit my daughter anymore. Very disappointing since I thought that this would be a good deal.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0138', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'No support for Windows XP. The software for this unit does not support Windows XP. The unit is not of much use without being able to interface with the computer.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0138', 'product_id': 'PROD_APP_SMOKE', 'text': "weird fit, majorly leaks overnight. These diapers have a weird high-cut fit, and our 2 1/2 year old has peed through them ENORMOUSLY both of the first two nights he wore them. Wet down to his feet. They just don't seem to have enough absorbency for the job.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0139', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'not worth it. This case offers very little protection, and most of the phone is exposed. Not worth it...', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0139', 'product_id': 'PROD_APP_SMOKE', 'text': 'good enough diapers for me. I have no problem with these diapers on my child, however, I wish I had ordered a smaller size because he happened to transition to size 3 shortly after this purchase. Just bad timing. Otherwise, great product!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0140', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Bluetooth. This Bluetooth does not work that well. It seems to understand my commands only part of the time. I will opt to buy a more expensive one with noise reduction too.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0140', 'product_id': 'PROD_APP_SMOKE', 'text': "Leaks. I've had to throw at least 6 onesies because of the leakage riding up my sons back:( awful my next attempt is to try pampers swaddles while he still fits into their size. Hopefully they work out since there are many good reviews.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0141', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Excellent Compilation. This compilation of early (31-34) Calloway features excellent sound quality and a first-rate band backing up the King of Jive. At a great bargain price, this set is recommended for those who like their 30s swing served hot.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0141', 'product_id': 'PROD_APP_SMOKE', 'text': 'DO NOT last the night. I bought these based on the reviews and the price. Big mistake. I only get 6 hrs of "dry" time before my little one soaks through his clothes and leaves a big wet spot on the bed.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0142', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Too old. This item is so old you need a network adapter or a phone line to operate. Still having problems getting signal to watch tv.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0142', 'product_id': 'PROD_APP_SMOKE', 'text': 'To small. I was using swaddlers (pampers yellow box) and then decided to try these baby-dry (pampers green box. My 5 month old is in a size 2 but when I ordered size 2 in the baby-dry they were to small even though she was still in the weight range (15lbs) they give on the box. So my advise is that the baby-dry pampers run small.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0143', 'product_id': 'PROD_ELEC_SMOKE', 'text': "tivo. I gave it a 1-star review because I couldn't get it to work. The Dish Network package where I live wouldn't allow me to connect with the TiVo service. Before ordering this dvr, always make sure it will work with your cable or sattelite provider.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0143', 'product_id': 'PROD_APP_SMOKE', 'text': 'Best diaper out there!. For my baby shower we did a diaper raffle, so I received every diaper brand out there. These are by far the best out there. My little boy is 5 months old and is in size 3. Never any leakage problems even through the night. All my friends swear by them too!!!', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0144', 'product_id': 'PROD_ELEC_SMOKE', 'text': "really really bad, worst video ever. Worst video ever!!!!! Don't waste your money. Poor poor video quality, people walk in front of the camera shots. The video has bird, frog and other various sounds that have nothing to do with the bird or animal on the poor quality video. You won't like it, your cat won't like it, and that is an understatement.", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0144', 'product_id': 'PROD_APP_SMOKE', 'text': 'Terrible. I have been usage the Size 2 for my son and he is still breastfeeding and so his BMs are still pretty runny. These diapers were terrible!- the poop leaked straight THROUGH the diapers- not leaking out the sides- THROUGH! That meant that there was no way that he could stay clean through a BM! Which is very annoying.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0145', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'great help!!!!!!!!!!. I think this software is a help because you can learn more languages. It is important to know other languages in the buisness woeld so you can talk to all clients.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0145', 'product_id': 'PROD_APP_SMOKE', 'text': 'Pampers Baby Dry. Love love them,my son goes 6hrs especially at night without a leak.These pampers are economical as he started using a size 3 at 5&1/2months and is now going to 9months yet he has not outgrown them.Would recommend highly.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0146', 'product_id': 'PROD_ELEC_SMOKE', 'text': '6 foot not 6 inches. Amazon sent wrong cable to me twice... the picture and description clearly show a six inch cable. Original order and replacement were wrong.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0146', 'product_id': 'PROD_APP_SMOKE', 'text': "Horrible - how does anyone use these???. I assumed since these are a well-known brand, that they would work well. They are the worst diapers I have ever used - pee leaks stright through them, ie if you don't change the baby pretty often, their clothes will be soaked more than once a day. How am I going to use up these 200+ diapers? Ugh.", 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0147', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'I will never buy another HP printer. Please do not waste your money. I have had one for 2 years and have reloaded the software to make it work at least 30 times.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0147', 'product_id': 'PROD_APP_SMOKE', 'text': 'Works but messy!. Bonds sole of shoes well, but it was difficult not to get the glue on other parts not intended to be glued.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0148', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'bootleg. This is a bootleg. There are legal releases with much better sound quality. Buy Grooving Kingston 12 (Hip-O Records), Completer Wailers parts 2 and 3 (JAD Records) or African Herbsman (Trojan Records) instead.', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0148', 'product_id': 'PROD_APP_SMOKE', 'text': 'Works GREAT!. My tennis shoe sole was peeling. I applied some shoe goo and the problem was solved. I glued the sole back on and let it dry 24 hours. Next day the tennis shoe was good as new.', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0149', 'product_id': 'PROD_ELEC_SMOKE', 'text': 'Steven Wilson is GOD!. Absolutely, mind-blowingly awesome! Steven Wilson is the master of multi-channel sound production. Only HE could put together a disc that can (almost) capture the experience of a Porcupine Tree concert. Not to be missed!', 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0149', 'product_id': 'PROD_APP_SMOKE', 'text': 'Shoo Goo. We always use this to support shoes that we love. Glad to still find it somewhere. The price is also right', 'vertical': 'apparel_fashion'}, {'review_id': 'smk_elec_0150', 'product_id': 'PROD_ELEC_SMOKE', 'text': "Satisfied. The camera worked great for about 3 weeks and then took a dump. Since this was such an expensive purchase, we contacted the seller and he was more than happy to help us with whatever was necessary to make it right. It's true...satisfaction is the key to success!", 'vertical': 'electronics_tech'}, {'review_id': 'smk_app_0150', 'product_id': 'PROD_APP_SMOKE', 'text': "Temperary at best. This held the rubber to the bottom of my tennis shoes for about 2 weeks. After re-applying this multiple times I'v decided to seek something that works longer term.", 'vertical': 'apparel_fashion'}]

elec_kw = re.compile(
    r'\b(laptop|computer|mac|pc|charger|charging|battery|phone|smartphone|screen|display|cable|cord|sound|audio|speaker|bluetooth|usb|wifi|wi-fi|hard drive|ssd|ram|memory|camera|tv|television|headphone|headphones|earbuds|earphones|keyboard|mouse|software|driver|device|monitor|electronics)\b',
    re.I
)
app_kw = re.compile(
    r'\b(shirt|t-shirt|tshirt|pants|jeans|dress|jacket|coat|shoes|shoe|boots|boot|sweater|hoodie|fabric|cotton|polyester|silk|wool|material|size|sizes|sizing|fit|fits|fitting|wear|wearing|sleeve|sleeves|collar|waist|stitching|stitched|cloth|clothes|clothing|apparel|zipper|seam|skirt|blouse|denim|leather|sneakers|heel|heels|sock|socks|hat|cap|glove|gloves|scarf|pockets|pocket|stretch|tight|loose|baggy|comfortable|stylish)\b',
    re.I
)
ood_kw = re.compile(
    r'\b(album|song|songs|cd|vinyl|track|tracks|movie|dvd|film|book|books|author|novel|read|reading|chapter|recipe|taste|flavor|scent|perfume|fragrance|smell|food|burger|pizza|salad|soup|dinner|lunch|breakfast|waiter|waitress|restaurant)\b',
    re.I
)

pool_elec = []
pool_app = []
seen_texts = set()

def add_texts(texts, target_pool, vert):
    for t in texts:
        t_clean = str(t).strip()
        if len(t_clean) < 20 or len(t_clean) > 1200 or t_clean in seen_texts or ood_kw.search(t_clean):
            continue
        seen_texts.add(t_clean)
        target_pool.append({"text": t_clean, "vertical": vert})

parquet_candidates = list(Path("/kaggle/input").rglob("*.parquet"))
print(f"Found {len(parquet_candidates)} parquet candidate files in /kaggle/input")
for p in parquet_candidates:
    p_str = str(p).lower()
    if "ood_fixture" in p_str or "mams" in p_str:
        continue
    try:
        df_p = pd.read_parquet(p)
        if "text" not in df_p.columns:
            continue
        is_app_source = "apparel" in p_str or ("domain" in df_p.columns and (df_p["domain"] == "apparel").any())
        is_elec_source = "tech_laptop" in p_str or "television" in p_str or ("domain" in df_p.columns and (df_p["domain"] == "tech").any())

        for _, row in df_p.iterrows():
            t = row["text"]
            t_clean = str(t).strip()
            if len(t_clean) < 20 or len(t_clean) > 1200 or t_clean in seen_texts or ood_kw.search(t_clean):
                continue

            if is_app_source and ("domain" not in df_p.columns or row.get("domain") == "apparel" or "apparel" in p_str):
                seen_texts.add(t_clean)
                pool_app.append({"text": t_clean, "vertical": "apparel_fashion"})
            elif is_elec_source and ("domain" not in df_p.columns or row.get("domain") in ("tech", "home") or "laptop" in p_str or "television" in p_str):
                seen_texts.add(t_clean)
                pool_elec.append({"text": t_clean, "vertical": "electronics_tech"})
            else:
                has_e = bool(elec_kw.search(t_clean))
                has_a = bool(app_kw.search(t_clean))
                if has_e and not has_a:
                    seen_texts.add(t_clean)
                    pool_elec.append({"text": t_clean, "vertical": "electronics_tech"})
                elif has_a and not has_e:
                    seen_texts.add(t_clean)
                    pool_app.append({"text": t_clean, "vertical": "apparel_fashion"})
    except Exception as e:
        print(f"Note reading {p.name}: {e}")

print(f"Available filtered pools from disk: {len(pool_elec)} Electronics, {len(pool_app)} Apparel")

# Ensure full 5,000 target with fallback top-up
if len(pool_app) < HALF_TARGET:
    print(f"⚡ Topping up Apparel pool from embedded curated reviews...")
    for item in EMBEDDED_SMOKE_REVIEWS:
        if item.get("vertical") == "apparel_fashion" and item.get("text") not in seen_texts:
            seen_texts.add(item["text"])
            pool_app.append({"text": item["text"], "vertical": "apparel_fashion"})

selected_elec = pool_elec[:HALF_TARGET]
selected_app = pool_app[:HALF_TARGET]

balanced_reviews = []
for idx in range(HALF_TARGET):
    if idx < len(selected_elec):
        r = dict(selected_elec[idx])
        r["review_id"] = f"slv_elec_{idx+1:05d}"
        r["product_id"] = f"PROD_ELEC_{idx+1:05d}"
        balanced_reviews.append(r)
    if idx < len(selected_app):
        r = dict(selected_app[idx])
        r["review_id"] = f"slv_app_{idx+1:05d}"
        r["product_id"] = f"PROD_APP_{idx+1:05d}"
        balanced_reviews.append(r)

df_source = pd.DataFrame(balanced_reviews, columns=["review_id", "product_id", "text", "vertical"])
print(f"✅ Balanced dataset assembled: {len(df_source)} reviews ({sum(df_source['vertical'] == 'electronics_tech')} Electronics, {sum(df_source['vertical'] == 'apparel_fashion')} Apparel)")

# -------------------------------------------------------------------------
# 2. Resumability & Checkpointing
# -------------------------------------------------------------------------
def get_processed_ids(out_dir: Path) -> Set[str]:
    processed = set()
    for pf in out_dir.glob("*.parquet"):
        try:
            df = pd.read_parquet(pf, columns=["review_id"])
            processed.update(df["review_id"].astype(str).tolist())
        except Exception:
            pass
    return processed

processed_ids = get_processed_ids(OUTPUT_DIR)
if processed_ids:
    print(f"🔄 Checkpoint detected: Found {len(processed_ids)} already processed review_ids across parquets.")
    df_source = df_source[~df_source["review_id"].astype(str).isin(processed_ids)].reset_index(drop=True)
    print(f"Remaining unprocessed reviews: {len(df_source)}")

if len(df_source) == 0:
    print("🎉 All 5,000 reviews have already been processed! Done.")
    sys.exit(0)

# -------------------------------------------------------------------------
# 3. 2-GPU Data Parallel Dispatch
# -------------------------------------------------------------------------
num_gpus = torch.cuda.device_count()
print(f"\n🎯 Hardware Dispatch: Detected {num_gpus} CUDA GPU(s).")

t_run_start = time.perf_counter()

if num_gpus >= 2:
    print("🚀 2× Tesla T4 detected! Dispatching 50/50 disjoint shards to Worker 0 (GPU 0) and Worker 1 (GPU 1)...")
    shard_0 = df_source.iloc[0::2].reset_index(drop=True)
    shard_1 = df_source.iloc[1::2].reset_index(drop=True)

    shard0_path = Path("/kaggle/working/shard_0.parquet")
    shard1_path = Path("/kaggle/working/shard_1.parquet")
    shard_0.to_parquet(shard0_path, index=False)
    shard_1.to_parquet(shard1_path, index=False)
    print(f"   • Shard 0: {len(shard_0)} reviews -> {shard0_path}")
    print(f"   • Shard 1: {len(shard_1)} reviews -> {shard1_path}")

    cmd0 = [
        sys.executable, "-u", "/kaggle/working/stage2_silver_worker.py",
        "--worker-id", "0",
        "--gpu-id", "0",
        "--shard-file", str(shard0_path),
        "--output-dir", str(OUTPUT_DIR),
        "--prefix", "gpu0_batch",
        "--batch-size", str(BATCH_SIZE),
        "--max-tokens", str(MAX_TOKENS),
    ]
    cmd1 = [
        sys.executable, "-u", "/kaggle/working/stage2_silver_worker.py",
        "--worker-id", "1",
        "--gpu-id", "1",
        "--shard-file", str(shard1_path),
        "--output-dir", str(OUTPUT_DIR),
        "--prefix", "gpu1_batch",
        "--batch-size", str(BATCH_SIZE),
        "--max-tokens", str(MAX_TOKENS),
    ]

    env0 = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
    env1 = dict(os.environ, CUDA_VISIBLE_DEVICES="1")

    p0 = subprocess.Popen(cmd0, env=env0, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, env=env1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def stream_worker_logs(proc, label):
        for line in iter(proc.stdout.readline, ''):
            if line:
                print(f"[{label}] {line.rstrip()}", flush=True)
        proc.stdout.close()

    t0 = threading.Thread(target=stream_worker_logs, args=(p0, "GPU 0"))
    t1 = threading.Thread(target=stream_worker_logs, args=(p1, "GPU 1"))
    t0.start()
    t1.start()
    t0.join()
    t1.join()

    ret0 = p0.wait()
    ret1 = p1.wait()
    assert ret0 == 0, f"Worker 0 failed with exit code {ret0}"
    assert ret1 == 0, f"Worker 1 failed with exit code {ret1}"
    print("✅ Both GPU 0 and GPU 1 workers finished cleanly!")
else:
    print("⚡ Single GPU detected. Running single worker on GPU 0...")
    shard_path = Path("/kaggle/working/shard_0.parquet")
    df_source.to_parquet(shard_path, index=False)
    cmd0 = [
        sys.executable, "-u", "/kaggle/working/stage2_silver_worker.py",
        "--worker-id", "0",
        "--gpu-id", "0",
        "--shard-file", str(shard_path),
        "--output-dir", str(OUTPUT_DIR),
        "--prefix", "gpu0_batch",
        "--batch-size", str(BATCH_SIZE),
        "--max-tokens", str(MAX_TOKENS),
    ]
    env0 = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
    p0 = subprocess.Popen(cmd0, env=env0, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(p0.stdout.readline, ''):
        if line:
            print(f"[GPU 0] {line.rstrip()}", flush=True)
    ret0 = p0.wait()
    assert ret0 == 0, f"Worker failed with exit code {ret0}"

total_wall_clock_sec = time.perf_counter() - t_run_start

# -------------------------------------------------------------------------
# 4. Final Aggregation & Performance Report
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("📊 ProductLens 3.0: Stage 2 5,000-Review Silver Run Aggregation")
print("=" * 72)

all_parquet_files = sorted(OUTPUT_DIR.glob("*.parquet"))
print(f"Total parquet files generated: {len(all_parquet_files)}")

dfs = []
for pf in all_parquet_files:
    try:
        dfs.append(pd.read_parquet(pf))
    except Exception as e:
        print(f"Warning reading {pf.name}: {e}")

if dfs:
    df_silver = pd.concat(dfs, ignore_index=True)
else:
    df_silver = pd.DataFrame()

total_quads = len(df_silver)
unique_reviews_labeled = df_silver["review_id"].nunique() if not df_silver.empty else 0
sec_per_500 = (total_wall_clock_sec / max(1, TARGET_REVIEWS)) * 500

print(f"Total Unique Reviews in Dataset:  {TARGET_REVIEWS}")
print(f"Total Unique Reviews Labeled:    {unique_reviews_labeled}")
print(f"Total Consensus Quads Kept:      {total_quads}")
print(f"Average Quads per Review:        {(total_quads / max(1, unique_reviews_labeled)):.2f}")
print(f"Total Wall-Clock Time:           {total_wall_clock_sec:.1f}s ({total_wall_clock_sec / 3600:.2f} hours)")
print(f"Throughput (sec / 500 revs):     {sec_per_500:.2f}s")
if not df_silver.empty and "vertical" in df_silver.columns:
    print(f"Quads by Vertical:               {dict(df_silver['vertical'].value_counts())}")
if not df_silver.empty and "sentiment" in df_silver.columns:
    print(f"Quads by Sentiment:              {dict(df_silver['sentiment'].value_counts())}")
print("=" * 72)

results_summary = {
    "target_reviews": TARGET_REVIEWS,
    "unique_reviews_labeled": unique_reviews_labeled,
    "total_consensus_quads": total_quads,
    "total_wall_clock_sec": round(total_wall_clock_sec, 2),
    "total_wall_clock_hours": round(total_wall_clock_sec / 3600, 2),
    "seconds_per_500_reviews": round(sec_per_500, 2),
    "parquet_files_count": len(all_parquet_files),
    "output_dir": str(OUTPUT_DIR),
}

with open("/kaggle/working/silver_5000_results.json", "w", encoding="utf-8") as f:
    json.dump(results_summary, f, indent=2)

print("✅ Saved summary to /kaggle/working/silver_5000_results.json")
